# Video Face Swap - 2 người (cảnh hôn) + **nhuộm tóc riêng cho từng người**

**Logic:** 1 video có 2 người + 2 ảnh khuôn mặt (A, B) → video mới giữ nguyên chuyển động/nền
gốc, **thay mặt đúng người** và **đổi màu tóc đúng người**.

Notebook này ghép hai bản đã có:

| Lấy từ | Phần nào |
|---|---|
| `video_face_swap_kiss.ipynb` | `FaceTracker` (khớp danh tính ArcFace + gán tối ưu toàn cục), `swap_all_faces` (parsenet + tranh chấp pixel + chừa mũi/môi lúc chạm nhau), hai cell chẩn đoán |
| `video_face_swap_1nguoi_mau_toc.ipynb` | `HairRecolor` (segment MediaPipe + đổi màu trong LAB + mép tóc + tóc mai + kết cấu sợi) |

## Việc MỚI phải giải: một mask tóc, hai người

MediaPipe `selfie_multiclass_256x256` trả về **một mask tóc cho cả khung hình**. Nó không biết
đâu là tóc A, đâu là tóc B — mà hai người mỗi người một màu tóc thì phải biết.

Cách giải: **chia mask theo khoảng cách tới mặt từng người**, chuẩn hoá theo bề ngang mặt (mặt
gần camera hơn thì to hơn, đáng được nhận vùng rộng hơn). Ranh giới có dải chuyển tiếp mềm
(`HAIR_SPLIT_BAND`) nên chỗ tóc hai người đan vào nhau không thành đường cắt gắt.

Phép chia chạy ở **không gian mask 256×256** (65k điểm, gần như miễn phí) rồi mới phóng lên
kích thước video — không phải tính khoảng cách cho từng pixel của khung 1080p.

Vẫn **chỉ một lần gọi MediaPipe cho mỗi frame**: segment một lần, chia mask, rồi đổi màu hai
lần với hai màu đích và **hai bộ thống kê riêng** (tóc A có thể tối hơn tóc B, mỗi người phải
được đo riêng nếu không màu ra sẽ lệch).

## Đường đi của một khung hình

```
frame gốc
  → app.get()            → detect mọi mặt
  → tracker.step()       → mặt nào là A, mặt nào là B (embedding ArcFace, không phải vị trí)
  → swap_all_faces()     → swap CẢ HAI cùng lúc từ frame gốc, không ai đè lên ai
  → GFPGAN               → làm nét từng mặt đã swap  (tuỳ chọn)
  → NHUỘM TÓC:  segment 1 lần → chia mask A/B → đổi màu từng người   ← phần mới
  → ffmpeg
```

**Nhuộm tóc chạy CUỐI CÙNG.** Ô crop của GFPGAN nới bbox thêm 40% nên lấn sang cả tóc; nhuộm
trước thì GFPGAN vẽ lại chính vùng tóc vừa nhuộm, mỗi frame một kiểu → nhấp nháy.

## Khác biệt so với bản 1 người

- Tóc **không phụ thuộc vào việc swap được mặt**: frame nào tracker không gán được người nào
  thì mặt giữ nguyên, nhưng tóc vẫn đổi màu — miễn là biết mặt người đó **ở đâu**.
- Mất mặt tạm thời (che, quay đi) vẫn chia mask được nhờ **vị trí lần thấy cuối** mà
  `FaceTracker` đang giữ. Có `HAIR_REACH` chặn bán kính để không nhuộm lan sang người thứ ba.

⚠️ Face-swap + hair-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà không có
sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý.

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

## 0. Cấu hình

| | `True` | `False` |
|---|---|---|
| `USE_GFPGAN` | cài `gfpgan`/`basicsr` (phải vá source), tải `GFPGANv1.4.pth` (~340MB), làm nét từng mặt đã swap | không cài, không tải |
| `USE_HAIR` | cài `mediapipe`, tải `selfie_multiclass_256x256.tflite` (~16MB), nhuộm tóc mỗi frame | giữ nguyên màu tóc trong video |

`facexlib` **luôn được cài** (không phụ thuộc `USE_GFPGAN`) vì `swap_all_faces` cần `parsenet`
để biết pixel nào là mặt ai lúc hai người áp má.

### Ai là A, ai là B

- **A = người bên TRÁI khung hình, B = người bên PHẢI** — quy ước lúc khởi tạo danh tính.
- Sau frame đầu, việc gán dựa trên **embedding ArcFace** chứ không phải vị trí, nên hai người
  đổi chỗ giữa video vẫn theo đúng người.
- Upload ảnh đúng thứ tự: ảnh A trước, ảnh B sau. Nếu ra kết quả đảo mặt, chỉ cần **đổi thứ tự
  hai ảnh** rồi chạy lại — không cần sửa code.

### Màu tóc: mỗi người một biến

`HAIR_COLOR_A` và `HAIR_COLOR_B`, mỗi cái nhận ba kiểu giá trị:

1. **`'ảnh'`** — lấy màu tóc của **chính người trong ảnh nguồn tương ứng** (A lấy từ ảnh A,
   B lấy từ ảnh B). Hợp lý nhất khi muốn video giống hệt hai người trong ảnh.
2. **Tên trong bảng màu** — `'nâu hạt dẻ'`, `'vàng đồng'`, `'đỏ rượu'`...
3. **Mã hex** — `'#8B5A2B'`

Đặt `None` cho người nào **không** muốn đổi màu tóc (giữ nguyên tóc gốc của người đó trong
video, vẫn swap mặt bình thường).

### Hai nút riêng của bản 2 người

- **`HAIR_SPLIT_BAND`**: độ mềm của ranh giới chia tóc A/B, tính theo bề ngang mặt. Nhỏ = cắt
  dứt khoát (dễ thấy đường ranh khi tóc hai người đan nhau), lớn = chuyển màu dần.
- **`HAIR_REACH`**: tóc xa hơn ngần này lần bề ngang mặt thì không nhuộm. Chặn trường hợp
  nhuộm lan sang người thứ ba trong khung, hoặc lan ra tóc người B khi B đang không detect
  được.

Còn lại là các nút giống bản 1 người. **Không cần hiểu hết ngay**: chạy tới mục 6.1 (thử 1
frame) sẽ thấy ngay kết quả kèm ảnh tô màu **vùng tóc của ai**.

In [ ]:
# ===================== CÔNG TẮC CHÍNH =====================
USE_GFPGAN = True    # True  = swap xong làm nét từng mặt (đẹp hơn, chậm hơn)
USE_HAIR   = True    # True  = đổi màu tóc của cả hai người
# ==========================================================


# ===================== MÀU TÓC CHO TỪNG NGƯỜI =====================
# Mỗi biến nhận: 'ảnh' (lấy màu tóc từ ảnh nguồn của CHÍNH người đó)
#              | tên trong HAIR_PALETTE | mã hex '#RRGGBB'
#              | None (không đổi màu tóc người này, vẫn swap mặt)
#
# A = người bên TRÁI khung hình lúc khởi tạo, B = bên PHẢI.
HAIR_COLOR_A = 'ảnh'
HAIR_COLOR_B = 'ảnh'

HAIR_PALETTE = {
    'đen':            '#1B1917',
    'nâu đen':        '#2E211B',
    'nâu socola':     '#43291B',
    'nâu hạt dẻ':     '#5A3220',
    'nâu tây':        '#8B5A2B',
    'nâu khói':       '#6B5A4E',
    'vàng đồng':      '#A9762F',
    'vàng mật ong':   '#C08D4A',
    'vàng bạch kim':  '#D8C8A2',
    'bạch kim':       '#CFC7BB',
    'trắng':          '#E5E3DE',
    'bạc':            '#9A9A9A',
    'đỏ rượu':        '#5E1F22',
    'đỏ cam':         '#8E3B1E',
    'hồng khói':      '#9C6A6E',
    'tím khói':       '#4A3350',
    'xanh rêu':       '#3A4232',
    'xanh khói':      '#37474F',
}
# ==================================================================


# ============ CHIA TÓC CHO A / B (chỉ có ở bản 2 người) ============
HAIR_SPLIT_BAND = 0.30   # độ mềm của ranh giới chia tóc, tính theo bề ngang mặt.
                         # MediaPipe chỉ cho MỘT mask tóc cho cả khung, không biết đâu là tóc
                         # ai; ta chia theo khoảng cách tới mặt từng người. Dải này quyết định
                         # chỗ giao nhau chuyển màu dần hay cắt dứt khoát.
                         # Tóc hai người đan vào nhau mà thấy đường ranh -> tăng (0.5).
                         # Màu người này lấn sang tóc người kia -> giảm (0.15).

HAIR_REACH      = 4.5    # tóc xa hơn ngần này lần bề ngang mặt thì KHÔNG nhuộm.
                         # Chặn hai tình huống: (1) trong khung có người thứ ba, (2) một người
                         # đang không detect được nên toàn bộ tóc dồn cho người còn lại.
                         # Tóc rất dài bị hụt màu ở phần cuối -> tăng (6.0).
# ===================================================================


# ============ TINH CHỈNH PHẦN NHUỘM (dùng chung cho cả A và B) ============
# Chỉnh xong chạy lại cell 6.1 (thử 1 frame) để xem ngay, không cần chạy cả video.

HAIR_LIGHTNESS      = 0.85   # 0-1: kéo ĐỘ SÁNG về màu đích. NÚT QUAN TRỌNG NHẤT.
                             # 0 = giữ nguyên độ sáng gốc, chỉ đổi sắc màu.
                             # 1 = bám hoàn toàn độ sáng màu đích.
                             # Nhuộm tóc ĐEN (L~40) sang TRẮNG (L~230) mà để 0.55 thì tóc chỉ
                             # đi được nửa đường -> ra XÁM. Nhưng để đúng 1.0 với màu gần
                             # trắng thì trung bình bị đẩy sát 255, không còn chỗ cho sợi
                             # sáng -> tóc bệt. 0.85 là chỗ cân bằng.

HAIR_DETAIL         = 1.0    # Khuếch đại KẾT CẤU SỢI TÓC. Nút quyết định tóc nhìn như TÓC hay
                             # như QUÉT SƠN. Tóc đen có biến thiên sáng-tối cỡ 9-12; tóc vàng/
                             # bạch kim thật cỡ 20-35. Nhuộm đen -> sáng mà không khuếch đại
                             # thì được một mảng sáng đều, mắt không thấy sợi.
                             # Mức khuếch đại TỰ ĐỘNG theo độ lệch sáng.
                             # 0 = tắt. 1 = tự động. 1.5 = đậm sợi hơn (kèm nhiễu hạt).

HAIR_KEEP_TONE      = 0.45   # 0-1: giữ bao nhiêu biến thiên màu gốc. 0 = màu đích phẳng lì.

HAIR_KEEP_HIGHLIGHT = 0.55   # 0-1: chừa bao nhiêu phần MÀU ở vùng tóc bắt sáng (ánh phản chiếu
                             # trên tóc thật gần như không màu). Để cao quá thì đỉnh đầu và chỗ
                             # rẽ ngôi nhìn như còn sót màu tóc cũ.

HAIR_CONTRAST       = 1.00   # nhân biến thiên độ sáng ở mức BÓNG ĐỔ LỚN. Muốn rõ sợi thì dùng
                             # HAIR_DETAIL, không phải nút này.

HAIR_SATURATION     = 1.00   # nhân độ rực của màu đích.
HAIR_STRENGTH       = 1.00   # độ đậm tổng thể. <1 = pha với màu tóc gốc.

HAIR_CONF           = (0.35, 0.65)   # dốc mềm của mask: dưới 0.35 không nhuộm, trên 0.65 nhuộm
                             # hết. Màu lem sang nền/áo/vai -> nâng lên (0.5, 0.8).
                             # Rìa tóc chưa ăn màu -> hạ xuống (0.25, 0.5).

HAIR_FEATHER        = 2.0    # làm mềm mép mask (đơn vị: pixel ở không gian 256x256).

HAIR_WISPS          = 0.7    # 0-1: nhuộm cả SỢI TÓC MAI mảnh rủ trước mặt - thứ mà mask
                             # 256x256 không thấy nổi. Mắt/mũi/miệng/lông mày đã chừa sẵn.
                             # Nhuộm nhầm vào vệt tối trên mặt/cổ -> giảm.

HAIR_EDGE_SMART     = False  # Xử lý MÉP TÓC bằng chính độ sáng của ảnh thay vì chỉ dựa vào
                             # mask 256x256. Sạch mép nhất nhưng làm vùng nhuộm đặc hơn.
                             # Bật lên nếu còn thấy vệt sáng ở đường viền tóc.

HAIR_EDGE_CHOKE     = 0      # CO mask vào trong (cách chữa quầng sáng CŨ, thô hơn: ăn cả mép
                             # tóc thật nên hay để lại viền tối). Để 0.

HAIR_MASK_SMOOTH    = 0.5    # 0-1: làm mượt mask theo thời gian (EMA) cho đỡ nhấp nháy.
HAIR_PROTECT_FACE   = True   # trừ vùng da mặt ra khỏi mask, tránh nhuộm lem lên trán.
# ==========================================================================

LABELS = ('A', 'B')


def _describe_color(v, who):
    """Kiểm tra giá trị màu NGAY TẠI ĐÂY, trước khi cài/tải hàng trăm MB rồi mới nổ."""
    if v is None:
        return f'{who}: KHÔNG đổi màu tóc (giữ nguyên tóc gốc trong video)'
    if v == 'ảnh':
        return f'{who}: lấy màu tóc từ ẢNH NGUỒN {who} (tính ở mục 5b)'
    if v in HAIR_PALETTE:
        return f'{who}: {v!r} = {HAIR_PALETTE[v]}'
    if isinstance(v, str) and v.startswith('#'):
        return f'{who}: mã hex {v}'
    raise ValueError(
        f'Màu tóc của {who} = {v!r} không hợp lệ.\n'
        f"Dùng None, 'ảnh', mã hex '#RRGGBB', hoặc một trong: {', '.join(HAIR_PALETTE)}"
    )


print('USE_GFPGAN =', USE_GFPGAN, '| USE_HAIR =', USE_HAIR)
if USE_HAIR:
    print(' ', _describe_color(HAIR_COLOR_A, 'A'))
    print(' ', _describe_color(HAIR_COLOR_B, 'B'))
    if HAIR_COLOR_A is None and HAIR_COLOR_B is None:
        print('  (cả hai đều None -> không nhuộm ai; đặt USE_HAIR = False cho gọn)')
    print(f'  chia tóc: ranh giới mềm {HAIR_SPLIT_BAND}, bán kính {HAIR_REACH} bề ngang mặt')
    print(f'  độ sáng {HAIR_LIGHTNESS} | rõ sợi {HAIR_DETAIL} | giữ sắc gốc {HAIR_KEEP_TONE} '
          f'| tóc mai {HAIR_WISPS}')
else:
    print('-> giữ nguyên màu tóc của cả hai người.')

## 1. Cài đặt thư viện

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.

# facexlib LUÔN cài, không phụ thuộc USE_GFPGAN: swap_all_faces (mục 6.1b) cần `parsenet`
# của nó để biết pixel nào thuộc mặt ai lúc hai người áp má. Đây là khác biệt so với bản
# 1 người - ở đó facexlib chỉ là dependency của GFPGAN nên tắt được cùng nhau.
EXTRA_PKGS = 'facexlib' + (' gfpgan' if USE_GFPGAN else '')
print('Cài thêm:', EXTRA_PKGS)
!pip install -q onnxruntime-gpu {EXTRA_PKGS}

# libportaudio2: mediapipe import `sounddevice`, thư viện này ném OSError ngay lúc import nếu
# thiếu PortAudio trên máy -> `import mediapipe` chết dù package cài thành công. Ảnh Colab có
# lúc có lúc không, cài luôn cho chắc (vài trăm KB).
!apt-get -qq install -y ffmpeg libportaudio2 > /dev/null
print('Xong.')

### Cài `mediapipe` mà không đụng vào `cv2` (chỉ chạy khi `USE_HAIR = True`)

`pip install mediapipe` kéo theo **`opencv-contrib-python`**. Package đó chiếm đúng namespace
`cv2` mà `opencv-python` (Colab cài sẵn, insightface đang dùng) đang chiếm — cài đè lên nhau là
đúng cái lỗi `cv2` lẫn lộn mà notebook gốc đã ghi chú tránh ở cell trên. Nó cũng hay ghim lại
`numpy`/`protobuf`, làm hỏng ngược `onnxruntime` vừa cài.

Nên cell dưới cài `--no-deps` rồi **tự cài đúng những dependency thật sự cần** cho Image
Segmenter (`absl-py`, `attrs`, `flatbuffers`, `protobuf`, `jax` không cần). `cv2` thì dùng lại
bản Colab đã có.

Nếu bản `--no-deps` không import được, cell **tự fallback sang `pip install mediapipe` đầy đủ**
và in cảnh báo — vẫn chạy được, chỉ là môi trường bẩn hơn. Nếu cả hai đều hỏng thì
`HAIR_AVAILABLE = False` và pipeline chạy tiếp **không thay tóc** (giống cách `gfpgan` fallback),
chứ không làm chết cả notebook.

In [ ]:
import subprocess, sys, importlib

HAIR_AVAILABLE = False


def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-2500:])
    print(r.stderr[-2500:])
    return r.returncode


def try_import_mediapipe():
    """Import thử cả `mediapipe` lẫn đúng submodule Image Segmenter sẽ dùng.

    Import mỗi `mediapipe` là chưa đủ: `mediapipe.tasks.python.vision` mới là chỗ ném lỗi khi
    thiếu dependency, mà nó chỉ được nạp khi gọi tới -> phải thử ngay tại đây.
    """
    importlib.invalidate_caches()
    try:
        import mediapipe as mp
        from mediapipe.tasks.python import vision as _vision   # noqa: F401
        print('mediapipe OK, version:', getattr(mp, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception chứ không chỉ ImportError: thiếu PortAudio ném OSError,
        # lệch protobuf ném TypeError/AttributeError -> chỉ bắt ImportError thì cell crash.
        print(f'Chưa import được mediapipe: {type(e).__name__}: {e}')
        return False


if not USE_HAIR:
    print('USE_HAIR = False -> bỏ qua mediapipe (chỉ cần cho phần tóc).')
else:
    HAIR_AVAILABLE = try_import_mediapipe()

    if not HAIR_AVAILABLE:
        print('Cài mediapipe --no-deps (giữ nguyên cv2/numpy của Colab)...')
        pip('install', '-q', '--no-deps', 'mediapipe')
        # Dependency tối thiểu cho Tasks API. Không đụng numpy/opencv/protobuf-version.
        pip('install', '-q', 'absl-py', 'attrs', 'flatbuffers', 'sentencepiece', 'sounddevice')
        HAIR_AVAILABLE = try_import_mediapipe()

    if not HAIR_AVAILABLE:
        print()
        print('Bản --no-deps không chạy -> fallback: cài mediapipe đầy đủ.')
        print('CẢNH BÁO: bước này có thể cài đè opencv-contrib-python lên cv2 hiện có.')
        pip('install', 'mediapipe')
        HAIR_AVAILABLE = try_import_mediapipe()

        # Cài đè cv2 xong phải kiểm tra lại chính cv2 + onnxruntime, vì đó là thứ dễ vỡ nhất.
        for mod in ('cv2', 'onnxruntime'):
            try:
                m = importlib.import_module(mod)
                print(f'  {mod} vẫn OK, version:', getattr(m, '__version__', '?'))
            except Exception as e:
                print(f'  !! {mod} HỎNG sau khi cài mediapipe: {type(e).__name__}: {e}')
                print('     Runtime > Restart session rồi chạy lại từ đầu notebook.')

    if not HAIR_AVAILABLE:
        print()
        print('mediapipe không cài được -> sẽ chạy tiếp mà KHÔNG thay tóc.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

### Fix riêng cho `basicsr` (chỉ chạy khi `USE_GFPGAN = True`)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy
version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy
(thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng
chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

`basicsr` chỉ là dependency của GFPGAN, nên `USE_GFPGAN = False` thì cell này không làm gì cả.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json


def install_basicsr_patched():
    """Tải basicsr từ PyPI, vá bug PEP 667 trong setup.py, rồi cài từ source đã sửa."""
    os.makedirs('/tmp/basicsr_src', exist_ok=True)

    # Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
    # setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
    with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
        pkg_info = json.load(resp)

    sdist_url = None
    for url_info in pkg_info['urls']:
        if url_info['packagetype'] == 'sdist':
            sdist_url = url_info['url']
            break
    assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

    tar_name = sdist_url.split('/')[-1]
    tar_path = f'/tmp/basicsr_src/{tar_name}'
    urllib.request.urlretrieve(sdist_url, tar_path)
    print(f'Đã tải: {tar_name}')

    extract_dir = '/tmp/basicsr_build'
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace(...),
        # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
        root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
        assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
        root_name = root_names.pop()
        # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
        tar.extractall(extract_dir, filter='data')

    pkg_dir = os.path.join(extract_dir, root_name)
    setup_py_path = os.path.join(pkg_dir, 'setup.py')
    assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

    with open(setup_py_path, 'r') as f:
        content = f.read()

    # Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
    content = content.replace(
        "exec(compile(f.read(), version_file, 'exec'))",
        "exec(compile(f.read(), version_file, 'exec'), globals())"
    )
    content = content.replace(
        "return locals()['__version__']",
        "return globals()['__version__']"
    )

    with open(setup_py_path, 'w') as f:
        f.write(content)

    print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
    # sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
    # thay vì lệnh `pip` bất kỳ đứng đầu PATH.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
        capture_output=True, text=True
    )
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
    print('Cài basicsr thành công.')


if USE_GFPGAN:
    install_basicsr_patched()
else:
    print('USE_GFPGAN = False -> bỏ qua basicsr (chỉ là dependency của GFPGAN).')

## 2. Tải model

| Model | Khi nào tải | Dung lượng |
|---|---|---|
| `inswapper_128.onnx` | luôn luôn | ~530 MB |
| `GFPGANv1.4.pth` | `USE_GFPGAN = True` | ~340 MB |
| `selfie_multiclass_256x256.tflite` | `USE_HAIR = True` | ~16 MB |
| `parsing_parsenet.pth` | **facexlib tự tải** lúc gọi lần đầu ở mục 6 | ~85 MB |

`inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính sách,
nên cell dưới thử một mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ công rồi upload
vào `/content/models/`.

Model segment tải **thẳng từ `storage.googleapis.com/mediapipe-models`** — kho chính thức của
Google cho MediaPipe Tasks. Nó phân 6 lớp: `0 background, 1 hair, 2 body-skin, 3 face-skin,
4 clothes, 5 others`. Notebook này dùng **lớp 1 (hair)** để biết nhuộm chỗ nào và **lớp 3
(face-skin)** để trừ ra, khỏi nhuộm lem lên da.

Mỗi lần tải đều **kiểm tra dung lượng file**: `wget` coi cả trang 404 là "tải thành công", nên
không kiểm thì mãi tới lúc nạp model mới nổ với lỗi onnx/torch/tflite không nói gì về nguyên
nhân thật.

In [ ]:
import os, subprocess
os.makedirs('/content/models', exist_ok=True)

INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
GFPGAN_PATH    = '/content/models/GFPGANv1.4.pth'
SEGMENTER_PATH = '/content/models/selfie_multiclass_256x256.tflite'

INSWAPPER_URL = 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'
GFPGAN_URL    = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
# Kho model chính thức của MediaPipe Tasks (Google), không phải mirror cộng đồng.
SEGMENTER_URL = ('https://storage.googleapis.com/mediapipe-models/image_segmenter/'
                 'selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite')


# Dùng subprocess thay vì `!wget`: lệnh `!` nằm trong khối `if` phụ thuộc vào chi tiết
# transform của IPython, còn subprocess thì chạy giống nhau ở mọi môi trường và
# trả về returncode để kiểm tra.
def download(url, path, min_mb, hint):
    """Tải file rồi KIỂM TRA DUNG LƯỢNG.

    wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra.
    Nếu không, mãi tới cell load model mới nổ với lỗi onnx/torch/tflite rất khó đoán nguyên nhân.
    """
    print(f'Đang tải {os.path.basename(path)} ...')
    subprocess.run(['wget', '-q', '-O', path, url])
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK  {os.path.basename(path)}: {size_mb:.1f} MB')


download(INSWAPPER_URL, INSWAPPER_PATH, 200,
         'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')

if USE_GFPGAN:
    download(GFPGAN_URL, GFPGAN_PATH, 300, 'Kiểm tra lại link GitHub release của GFPGAN.')
else:
    print('Bỏ qua GFPGANv1.4.pth (~340MB) vì USE_GFPGAN = False.')

if USE_HAIR:
    download(SEGMENTER_URL, SEGMENTER_PATH, 5,
             'Kiểm tra lại đường dẫn trong kho mediapipe-models của Google.')
else:
    print('Bỏ qua selfie_multiclass_256x256.tflite vì USE_HAIR = False.')

# parsenet của facexlib (cho swap_all_faces) KHÔNG tải ở đây: facexlib tự tải weights của nó
# vào lần gọi init_parsing_model() đầu tiên, ở mục 6.
print()
!ls -lh /content/models/

## 3. Upload 2 ảnh khuôn mặt (người A, người B) + video mẫu (2 người)

In [ ]:
from google.colab import files

def pick_one(uploaded, what):
    names = list(uploaded.keys())
    assert len(names) > 0, f'Chưa upload {what} (bấm Cancel?). Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]

print('>> Upload ảnh khuôn mặt NGƯỜI A (rõ mặt, chính diện càng tốt):')
source_face_path_a = pick_one(files.upload(), 'ảnh mặt A')

print('\n>> Upload ảnh khuôn mặt NGƯỜI B:')
source_face_path_b = pick_one(files.upload(), 'ảnh mặt B')

print('\n>> Upload video mẫu (có 2 người, ví dụ cảnh hôn nhau):')
source_video_path = pick_one(files.upload(), 'video mẫu')

print(f'Ảnh mặt A: {source_face_path_a}')
print(f'Ảnh mặt B: {source_face_path_b}')
print(f'Video mẫu: {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại
`onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13),
sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn).

In [ ]:
import subprocess, sys, importlib

def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode

if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu cùng chiếm package `onnxruntime`. Cài cái này đè cái kia
    # là trạng thái hỏng đã biết (mất CUDAExecutionProvider, import lỗi loạn)
    # -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu==1.20.0')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import numpy as np
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Dùng providers:', providers, '| ctx_id =', ctx_id)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

swapper = insightface.model_zoo.get_model(INSWAPPER_PATH, download=False, providers=providers)


def load_source(path, label):
    """Trả về (ảnh, khuôn mặt). Khác bản kiss: giữ luôn cả ẢNH.

    Phần nhuộm tóc cần chính bức ảnh đó để đo màu tóc (HAIR_COLOR_x = 'ảnh'), nên đọc một
    lần rồi giữ, thay vì cell mục 5b phải imread lại và có thể đọc khác đường dẫn.
    """
    # cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
    # Phải chặn ngay, không thì app.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
    img = cv2.imread(path)
    assert img is not None, (
        f'Không đọc được ảnh {label} ({path}). Lưu lại thành .jpg/.png rồi upload lại.'
    )
    faces = app.get(img)
    assert len(faces) > 0, f'Không tìm thấy khuôn mặt trong ảnh {label}, thử ảnh khác rõ mặt hơn.'
    if len(faces) > 1:
        # Ảnh nguồn có nhiều mặt -> lấy mặt to nhất, vì thứ tự app.get() trả về không xác định.
        faces = sorted(faces,
                       key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]),
                       reverse=True)
        print(f'  (ảnh {label} có {len(faces)} mặt, dùng mặt lớn nhất)')
    return img, faces[0]


source_img_a, source_face_a = load_source(source_face_path_a, 'A')
source_img_b, source_face_b = load_source(source_face_path_b, 'B')

# Dùng ở mục 6: swap_all_faces tra theo nhãn.
source_face = {'A': source_face_a, 'B': source_face_b}

print('Đã detect khuôn mặt nguồn A và B thành công.')

# Cảnh báo sớm: hai ảnh nguồn giống nhau tới mức nào. FaceTracker phân biệt A/B bằng embedding
# của MẶT TRONG VIDEO nên chuyện này không làm nó lẫn, nhưng nếu bạn upload nhầm CÙNG MỘT ảnh
# hai lần thì kết quả là hai người cùng một mặt - rất dễ tưởng là lỗi tracking.
sim_ab = float(np.dot(source_face_a.normed_embedding, source_face_b.normed_embedding))
print(f'Độ giống nhau giữa hai ảnh nguồn: {sim_ab:.3f}', end='  ')
if sim_ab > 0.85:
    print('<- RẤT CAO: có phải bạn upload cùng một ảnh hai lần?')
elif sim_ab > 0.5:
    print('(khá giống - hai người nhìn na ná nhau, kết quả vẫn đúng)')
else:
    print('(hai người khác nhau rõ ràng)')

## 5. Làm nét mặt (GFPGAN) — cả mục này tùy thuộc `USE_GFPGAN`

Nếu bạn đặt `USE_GFPGAN = False` ở mục 0 thì **cứ chạy tuần tự cả 3 cell dưới**, chúng sẽ tự
in một dòng rồi bỏ qua. Không cần nhớ bỏ cell nào.

### Fix riêng cho `basicsr` (bug với `torchvision` mới trên Colab)

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi
`basicsr` vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay
nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ
không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi
file `.py` còn tham chiếu module cũ (trong cả `basicsr`, `facexlib`, `gfpgan`).

Nó cũng tự **xoá các module hỏng khỏi `sys.modules`** cả trước lẫn sau khi vá, nhờ vậy
**không cần Runtime > Restart session**.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó.

    find_spec() chỉ định vị package, không chạy __init__.py -> không dính đúng cái
    ModuleNotFoundError mà ta đang muốn vá.
    """
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


def patch_torchvision_refs():
    purge_modules()

    targets = {}
    for name in PKGS:
        d = package_dir(name)
        if d is None:
            print(f'{name:9s}: CHƯA CÀI')
        else:
            print(f'{name:9s}: {d}')
            targets[name] = d

    # Khác bản 1 người: ở đây facexlib được cài KỂ CẢ khi USE_GFPGAN = False (parsenet cần
    # nó), nên không được đòi phải có basicsr - chỉ đòi có ít nhất một package để vá.
    assert targets, (
        'Không tìm thấy basicsr/facexlib/gfpgan. Chạy lại cell cài đặt ở mục 1 rồi chạy lại.'
    )
    if USE_GFPGAN and 'basicsr' not in targets:
        raise AssertionError(
            'USE_GFPGAN = True nhưng không thấy basicsr. Chạy lại cell cài basicsr ở mục 1.'
        )

    patched = []
    for name, d in targets.items():
        for p in d.rglob('*.py'):
            try:
                text = p.read_text(encoding='utf-8')
            except (UnicodeDecodeError, OSError):
                continue
            if OLD_MOD not in text:
                continue
            p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
            patched.append(p)

    print()
    if patched:
        for p in patched:
            print(f'đã vá: {p}')
    else:
        print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn '
              'functional_tensor).')

    # Purge lần nữa sau khi vá, để lần import sau đọc lại file mới trên đĩa.
    purge_modules()

    # Kiểm chứng ngay tại đây thay vì để tới lúc chạy video mới biết. Kiểm cái thật sự cần:
    # parsenet của facexlib (luôn dùng), và basicsr chỉ khi bật GFPGAN.
    checks = [('facexlib.parsing', 'facexlib' in targets)]
    if USE_GFPGAN:
        checks.append(('basicsr.data.degradations', 'basicsr' in targets))
    for mod, should in checks:
        if not should:
            continue
        try:
            importlib.import_module(mod)
            print(f'OK: import {mod} thành công.')
        except Exception as e:
            print(f'VẪN LỖI khi import {mod}: {type(e).__name__}: {e}')
            print('Nếu lỗi liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại.')
            raise


patch_torchvision_refs()

In [ ]:
import subprocess, sys, importlib


def try_import_gfpgan():
    importlib.invalidate_caches()
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception: gfpgan hỏng vì basicsr/torchvision thường ném ModuleNotFoundError,
        # nhưng tuỳ phiên bản torch cũng có thể là AttributeError/OSError.
        print(f'Chưa import được gfpgan: {type(e).__name__}: {e}')
        return False


GFPGAN_AVAILABLE = False

if not USE_GFPGAN:
    print('USE_GFPGAN = False -> bỏ qua kiểm tra gfpgan.')
else:
    GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print('Thử cài lại gfpgan với log đầy đủ...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'gfpgan'],
                           capture_output=True, text=True)
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print()
        print('gfpgan không cài được -> sẽ tự động chạy tiếp mà KHÔNG làm nét.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

In [ ]:
# restorer = None nghĩa là "không làm nét". Vòng lặp ở mục 6 chỉ nhìn biến này,
# nên không cần kiểm tra USE_GFPGAN lần nữa ở trong đó.
restorer = None

if USE_GFPGAN and GFPGAN_AVAILABLE:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path=GFPGAN_PATH,
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('GFPGAN sẵn sàng (chỉ chạy trên vùng crop quanh mặt đã swap).')
elif USE_GFPGAN:
    print('USE_GFPGAN = True nhưng gfpgan không dùng được -> chạy tiếp, chỉ swap không làm nét.')
else:
    print('USE_GFPGAN = False -> chỉ swap, không làm nét.')

## 5b. Nhuộm tóc — một mask tóc, hai người

Cả mục này phụ thuộc `USE_HAIR`. Tắt thì hai cell dưới in một dòng rồi bỏ qua.

### Vấn đề riêng của bản 2 người

MediaPipe `selfie_multiclass_256x256` trả về **một mask tóc cho cả khung hình**. Nó phân biệt
tóc / da mặt / da người / áo / nền — chứ **không** phân biệt *tóc của ai*. Mà A và B mỗi người
một màu thì bắt buộc phải biết.

**Cách chia:** gán mỗi điểm cho người có mặt **gần hơn**, khoảng cách **chuẩn hoá theo bề ngang
mặt của chính người đó**. Chuẩn hoá là chỗ dễ bỏ sót: mặt gần camera thì to hơn và tóc trải
rộng hơn, nếu so khoảng cách thô thì người ngồi gần luôn bị thu hẹp vùng oan.

```
share_A = smooth(  d_B − d_A  )        d_x = khoảng cách tới mặt x / bề ngang mặt x
share_B = 1 − share_A
```

Ba chi tiết làm nó không vỡ:

- **Dải chuyển tiếp mềm** (`HAIR_SPLIT_BAND`): chỗ tóc hai người đan vào nhau, `share` chuyển
  dần từ 0 sang 1 nên màu pha vào nhau thay vì có một đường cắt gắt chạy qua giữa mảng tóc.
- **`share_A + share_B = 1`** ở mọi điểm: vùng tranh chấp được **pha** hai màu, không bị nhuộm
  hai lượt (nhuộm hai lượt thì lượt sau lấy kết quả lượt trước làm gốc → màu sai hẳn).
- **Chặn bán kính** (`HAIR_REACH`): tóc xa hơn ngần đó lần bề ngang mặt thì không nhuộm. Không
  có nó, frame nào B mất dấu là **toàn bộ** tóc trong khung dồn cho A — kể cả tóc B.

Phép chia chạy ở **không gian mask 256×256** (65 nghìn điểm) rồi mới phóng lên cỡ video, thay
vì tính khoảng cách cho 2 triệu pixel của khung 1080p.

### Mỗi người một bộ thống kê

Sau khi chia, phần đổi màu chạy **riêng cho từng người**. Đây không phải để cho gọn code:
công thức dịch độ sáng lấy **trung bình độ sáng của mái tóc** làm mốc, nên nếu tóc A tối và
tóc B sáng mà đo chung một mốc thì **cả hai đều lệch**. Trung bình L/a/b và độ lệch chuẩn đều
tính trên mask của riêng người đó.

Ngưỡng "thế nào là vùng bắt sáng" cũng theo phân bố của chính mái tóc đó — nên A tóc đen trong
bóng tối và B tóc vàng dưới đèn vẫn được xử lý đúng theo từng người.

### Người đặt màu `None` vẫn tham gia phép chia

Nếu chỉ nhuộm A và để B là `None`, B **vẫn phải góp mặt** vào bước chia. Bỏ B ra thì tóc B
trở thành "vùng gần A nhất" và ăn màu của A. Đây là lý do phép chia dùng danh sách *người có
vị trí*, còn việc nhuộm mới dùng danh sách *người có màu đích*.

### Phần còn lại giống bản 1 người

Đổi màu trong **LAB** (độ sáng tách rời màu sắc), giữ nguyên biến thiên sáng-tối của sợi tóc,
khuếch đại phần chi tiết để tóc không thành mảng phẳng, nén mềm hai đầu thay vì cắt cụt ở
0/255, chừa vùng bắt sáng, và một tầng riêng bắt **sợi tóc mai** rủ trước mặt — thứ mà mask
256×256 không thấy nổi.

In [ ]:
import numpy as np
import cv2

# Nhãn của model selfie_multiclass_256x256 (theo tài liệu MediaPipe).
LBL_BACKGROUND, LBL_HAIR, LBL_BODY_SKIN, LBL_FACE_SKIN, LBL_CLOTHES, LBL_OTHERS = range(6)

SEG_SIZE = 256   # model vốn chạy ở 256x256; đưa vào đúng cỡ này để nó không phải nội suy 2 lần


def mask_2d(arr):
    """Ép mask của MediaPipe về đúng 2D (H, W).

    Có bản mediapipe trả confidence mask shape (H, W, 1) thay vì (H, W). Trộn hai kiểu này
    lại là lỗi ngầm rất khó đoán: cv2.GaussianBlur/resize LẶNG LẼ bỏ chiều cuối, nên một mask
    thành 2D còn mask kia vẫn 3D, rồi phép nhân giữa chúng nổ
    "non-broadcastable output operand ... doesn't match the broadcast shape (256,256,256)"
    - hoặc tệ hơn, np.nonzero() trả 3 mảng thay vì 2. Chuẩn hoá ngay tại nguồn, một lần.
    """
    # copy=True là BẮT BUỘC, không phải cho chắc: np.asarray() trên mảng float32 đã liền khối
    # trả về CHÍNH mảng đó, tức là một view vào bộ nhớ MediaPipe - và bộ nhớ đó bị ghi đè ở lần
    # segment() sau, làm mask của frame trước đổi giá trị sau lưng mình (EMA thành vô nghĩa).
    a = np.array(arr, dtype=np.float32, copy=True)
    if a.ndim == 3 and a.shape[2] == 1:
        a = a[:, :, 0]              # view vào BẢN COPY ở trên -> vẫn an toàn
    assert a.ndim == 2, f'Mask của MediaPipe có shape lạ: {a.shape}'
    return np.ascontiguousarray(a)


def hex_to_bgr(s):
    s = s.strip().lstrip('#')
    assert len(s) == 6, f'Mã màu phải dạng #RRGGBB, nhận được {s!r}'
    r, g, b = int(s[0:2], 16), int(s[2:4], 16), int(s[4:6], 16)
    return np.uint8([[[b, g, r]]])


def bgr_to_hex(bgr):
    b, g, r = [int(v) for v in bgr]
    return f'#{r:02X}{g:02X}{b:02X}'


def to_lab(bgr_1px):
    """Một pixel BGR -> (L, a, b) trong thang uint8 của OpenCV (L 0-255, a/b lệch 128)."""
    return cv2.cvtColor(bgr_1px, cv2.COLOR_BGR2LAB)[0, 0].astype(np.float32)


class HairRecolor:
    """Đổi màu tóc cho NHIỀU người trong cùng một frame: segment 1 lần -> chia mask -> nhuộm.

    Khác bản 1 người ở đúng một chỗ: MediaPipe trả về MỘT mask tóc cho cả khung hình, không
    biết đâu là tóc A đâu là tóc B. Phần `_split_by_person` giải việc đó; toàn bộ phần đổi
    màu bên dưới thì chạy riêng cho từng người với màu đích và thống kê riêng của người đó.
    """

    def __init__(self, model_path):
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision as mp_vision

        self._mp = mp
        common = dict(base_options=mp_python.BaseOptions(model_asset_path=model_path),
                      running_mode=mp_vision.RunningMode.IMAGE)
        try:
            options = mp_vision.ImageSegmenterOptions(
                **common,
                output_category_mask=False,   # chỉ cần confidence mask (float, mép mềm)
                output_confidence_masks=True,
            )
        except TypeError as e:
            # mediapipe < 0.10.3 dùng bộ tham số khác. Không ghim version ở mục 1 (dễ hết wheel
            # cho Python mới trên Colab) nên chấp nhận cả hai: _segment() tự xoay theo kết quả.
            print(f'  (ImageSegmenterOptions bản cũ: {e})')
            options = mp_vision.ImageSegmenterOptions(**common)
        self.segmenter = mp_vision.ImageSegmenter.create_from_options(options)

        self.prev_masks = None            # EMA theo thời gian (dùng chung, segment 1 lần)
        self.target_lab = {}              # {'A': (L,a,b), 'B': ...} - None/thiếu = không nhuộm
        self.target_bgr = {}
        self.last_alpha = {}              # {label: alpha} của frame gần nhất, để xem ở mục 6.1
        self.last_box = {}
        self.n_applied = {}

    # ---------------------------------------------------------------- segment
    def _segment(self, bgr):
        """Trả (hair, face_skin) - hai mask xác suất float32 cỡ SEG_SIZE x SEG_SIZE.

        Luôn đưa vào đúng SEG_SIZE, và luôn là CẢ KHUNG HÌNH. Chi phí segment cố định, không
        phụ thuộc độ phân giải video - và quan trọng với bản 2 người: MỘT lần gọi cho cả hai,
        không phải hai lần.
        """
        small = cv2.resize(bgr, (SEG_SIZE, SEG_SIZE), interpolation=cv2.INTER_AREA)
        rgb = np.ascontiguousarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
        mp_img = self._mp.Image(image_format=self._mp.ImageFormat.SRGB, data=rgb)
        res = self.segmenter.segment(mp_img)
        masks = getattr(res, 'confidence_masks', None)
        if masks:
            # mask_2d() vừa copy vừa ép về 2D - xem docstring của nó để biết vì sao cần cả hai.
            hair = mask_2d(masks[LBL_HAIR].numpy_view())
            face = mask_2d(masks[LBL_FACE_SKIN].numpy_view())
        else:
            # Bản mediapipe không cho confidence mask -> lấy category mask (0/1, mép gắt).
            # Vẫn chạy được vì bước sau còn làm mềm mép bằng Gaussian.
            cat = mask_2d(res.category_mask.numpy_view())
            hair = (cat == LBL_HAIR).astype(np.float32)
            face = (cat == LBL_FACE_SKIN).astype(np.float32)
        return hair, face

    # ------------------------------------------------------------- chọn màu
    def hair_color_of(self, img):
        """Màu tóc của một ảnh nguồn (cho HAIR_COLOR_x = 'ảnh'). Trả (bgr, tỉ lệ phủ).

        Lấy TRUNG VỊ chứ không phải trung bình: vùng tóc luôn dính vài pixel nền/da ở rìa,
        trung bình bị chúng kéo lệch, trung vị thì không.
        """
        h, w = img.shape[:2]
        hair, _ = self._segment(img)
        m = cv2.resize(hair, (w, h), interpolation=cv2.INTER_LINEAR) >= 0.6
        cover = float(m.mean())
        assert m.sum() > 200, (
            'Không tìm thấy vùng tóc trong ảnh nguồn để lấy màu.\n'
            'Dùng ảnh khác thấy rõ tóc, hoặc đặt màu bằng tên/mã hex thay cho "ảnh".'
        )
        return np.median(img[m], axis=0).astype(np.uint8), cover

    def set_target(self, label, bgr):
        if bgr is None:
            self.target_lab.pop(label, None)
            self.target_bgr.pop(label, None)
            return
        bgr = np.asarray(bgr, dtype=np.uint8).reshape(1, 1, 3)
        self.target_bgr[label] = bgr[0, 0].copy()
        self.target_lab[label] = to_lab(bgr)
        self.n_applied.setdefault(label, 0)

    def reset(self):
        self.prev_masks = None
        self.last_alpha = {}
        self.last_box = {}

    # --------------------------------------------------- chia tóc cho từng người
    @staticmethod
    def _split_by_person(m, people, sx, sy):
        """Chia MỘT mask tóc thành mask riêng cho từng người.

        MediaPipe cho một mask tóc cho cả khung; nó không biết đâu là tóc ai. Ta gán theo
        khoảng cách tới mặt từng người, CHUẨN HOÁ theo bề ngang mặt của chính người đó - mặt
        gần camera thì to hơn, tóc cũng trải rộng hơn, nên nếu chỉ so khoảng cách thô thì
        người ở gần luôn bị "thu hẹp" oan.

        Chạy ở không gian mask 256x256 (65 nghìn điểm) rồi mới phóng lên cỡ video, thay vì
        tính khoảng cách cho từng pixel của khung 1080p (2 triệu điểm).

        Trả {label: mask}. Tổng các mask đúng bằng mask gốc ở vùng hai người tranh nhau, nên
        chỗ giao là pha giữa hai màu, không phải nhuộm hai lần.
        """
        # Lưới toạ độ quy về PIXEL CỦA FRAME, không phải chỉ số ô mask. Mask 256x256 bị nén
        # méo với video không vuông (1280x720: một ô mask = 5 px theo x nhưng 2.8 px theo y),
        # nên đo khoảng cách bằng chỉ số ô là sai lệch 1.8 lần - ranh giới chia lệch hẳn và
        # HAIR_REACH chặn quá sớm theo chiều ngang.
        gx = (np.arange(SEG_SIZE, dtype=np.float32) * sx)[None, :]
        gy = (np.arange(SEG_SIZE, dtype=np.float32) * sy)[:, None]

        dist = {}
        for label, info in people.items():
            cx, cy = float(info['center'][0]), float(info['center'][1])
            wf = max(float(info['width']), 1e-6)
            dist[label] = np.sqrt((gx - cx) ** 2 + (gy - cy) ** 2) / wf

        out = {}
        labels = list(people)
        for label in labels:
            d = dist[label]
            if len(labels) > 1:
                # Gần mình hơn người khác bao nhiêu -> dốc mềm quanh mốc "bằng nhau".
                other = np.min([dist[o] for o in labels if o != label], axis=0)
                share = np.clip(0.5 + (other - d) / (2.0 * max(HAIR_SPLIT_BAND, 1e-6)), 0.0, 1.0)
            else:
                share = np.ones_like(d)
            # Chặn bán kính: tóc quá xa mặt thì không phải tóc người này. Không có cái này,
            # khi một người mất dấu thì toàn bộ tóc trong khung dồn cho người còn lại.
            reach = np.clip((HAIR_REACH - d) / max(0.25 * HAIR_REACH, 1e-6), 0.0, 1.0)
            out[label] = m * share * reach
        return out

    # ---------------------------------------------------------- tóc mai
    @staticmethod
    def _wisp_matte(L, cov, kps_local, d, L_hair):
        """Bắt những SỢI TÓC MAI mảnh rủ trước mặt, thứ mà mask 256x256 không thấy nổi.

        Sợi tóc mai rộng cỡ 4-8 pixel trên video 1080p, tức CHƯA TỚI MỘT Ô của mask 256x256 -
        ô đó bị da mặt phía sau lấn át nên xác suất "tóc" rất thấp. Tệ hơn, HAIR_PROTECT_FACE
        còn trừ thẳng vùng da mặt ra khỏi mask, mà tóc mai nằm đúng trên đó.

        Cách bắt: sợi tóc mai là VẠCH TỐI MẢNH trên nền da sáng hơn. Phép "black-hat" (đóng
        ảnh rồi trừ đi ảnh gốc) cho ra đúng những vạch tối mảnh hơn hạt nhân, và giá trị của
        nó chính là ĐỘ CHÊNH SÁNG giữa sợi tóc và da xung quanh. Chia cho chênh lệch tối đa
        (da - tóc) thì ra luôn tỉ lệ phủ của sợi trên pixel đó.

        Vùng mắt/mũi/miệng bị loại trừ: chúng cũng là "vạch tối trên nền da", nhuộm vào đó thì
        hỏng mặt. Lông mày nằm trong vùng loại trừ luôn, vì nhuộm lông mày hiếm khi là ý muốn.
        """
        h, w = cov.shape
        k = int(np.clip(0.18 * d, 9, 41)) | 1
        closing = cv2.morphologyEx(L, cv2.MORPH_CLOSE,
                                   cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        blackhat = closing - L                    # vạch càng tối so với xung quanh thì càng lớn

        # Trừ 6 làm ngưỡng nhiễu, nếu không thì vân da và nhiễu ảnh cũng bị nhận là tóc.
        denom = np.maximum(closing - L_hair, 15.0)
        wisp = np.clip((blackhat - 6.0) / denom, 0.0, 1.0)

        # Chỉ nhận sợi ở GẦN mái tóc CỦA NGƯỜI NÀY (cov đã là mask riêng của họ), nên tầng
        # tóc mai của A không thể ăn sang mặt B. Bán kính phải rộng: tóc mai rủ dài quá cằm,
        # lấy hẹp thì nhuộm được đoạn đầu rồi cụt giữa chừng - còn xấu hơn để nguyên.
        # Hạt nhân CHỮ NHẬT vì OpenCV làm nó tách rời theo hai trục, bán kính lớn vẫn rẻ.
        solid = (cov >= 0.5).astype(np.uint8)
        rn = max(3, int(2.0 * d))
        near = cv2.dilate(solid, cv2.getStructuringElement(cv2.MORPH_RECT, (2 * rn + 1, 2 * rn + 1)))
        wisp = wisp * cv2.GaussianBlur(near.astype(np.float32), (0, 0), sigmaX=max(1.0, 0.05 * d))

        le, re = kps_local[0], kps_local[1]
        mouth = (kps_local[3] + kps_local[4]) / 2.0
        cen = ((le + re) / 2.0 + mouth) / 2.0
        ang = float(np.degrees(np.arctan2(re[1] - le[1], re[0] - le[0])))
        feat = np.zeros((h, w), np.float32)
        cv2.ellipse(feat, (int(round(cen[0])), int(round(cen[1]))),
                    (int(1.05 * d), int(1.2 * d)), ang, 0, 360, 1.0, -1)
        feat = cv2.GaussianBlur(feat, (0, 0), sigmaX=max(1.0, 0.12 * d))
        return wisp * (1.0 - feat)

    # ------------------------------------------------------------ mép tóc
    @staticmethod
    def _edge_matte(L, cov, edge_px):
        """Ước lượng lại độ phủ tóc ở DẢI MÉP bằng chính độ sáng của ảnh.

        Mask MediaPipe chỉ 256x256; phóng lên 1080p thì mỗi ô mask thành hơn 4 pixel ảnh, và
        trong dải mép rộng cỡ đó mask KHÔNG biết pixel nào là tóc, pixel nào là nền. Chỉ dựa
        vào mask thì có đúng hai lựa chọn, đều tệ: nới rộng -> QUẦNG SÁNG ở nền; co hẹp ->
        VIỀN TỐI màu tóc cũ.

        Nhưng bức ảnh có thông tin đó: pixel ở mép là màu PHA giữa tóc và nền, nên vị trí của
        nó trên thang độ sáng chính là TỈ LỆ PHA. Nền được ước lượng CỤC BỘ quanh từng điểm,
        vì quanh đầu chỗ là tường, chỗ là vai áo, chỗ là mặt người kia.
        """
        h, wd = cov.shape
        core = (cov >= 0.9).astype(np.float32)
        n_core = float(core.sum())
        if n_core < 50:
            return cov              # tóc quá nhỏ/mảnh -> không đủ cơ sở, cứ tin mask

        L_hair = float((L * core).sum() / n_core)

        k = max(9, (int(0.15 * max(h, wd)) | 1))
        bg = (cov < 0.05).astype(np.float32)
        den = cv2.blur(bg, (k, k))
        L_bg = cv2.blur(L * bg, (k, k)) / np.maximum(den, 1e-3)

        d = L_hair - L_bg
        ok = np.abs(d) > 12.0       # tóc và nền sáng xấp xỉ nhau -> không tách được bằng độ sáng
        key = np.clip((L - L_bg) / np.where(ok, d, 1.0), 0.0, 1.0)
        key = np.where(ok, key, cov)                 # không tách được thì tin mask như cũ

        # Key CHỈ dùng ở VÀNH NGOÀI. Sâu bên trong mái tóc thì nhuộm đủ, KHÔNG hỏi key: chỗ
        # tóc bắt sáng sáng gần bằng nền nên key ở đó thấp, nghe theo nó là để lại nguyên
        # mảng màu cũ ngay giữa đỉnh đầu. "Sâu bên trong" = co vùng mask vào edge_px pixel,
        # chứ không phải "cov cao": mask hay lưỡng lự (0.5-0.7) ngay giữa mái tóc.
        r = max(1, int(edge_px))
        solid = (cov >= 0.5).astype(np.uint8)
        deep = cv2.erode(solid, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
        deep = cv2.blur(deep.astype(np.float32), (r | 1, r | 1))     # làm mềm bậc thang

        core_w = np.clip((cov - 0.85) / 0.15, 0.0, 1.0)
        support = np.clip(cov / 0.10, 0.0, 1.0)                      # chặn key lan ra ngoài mask
        return np.clip(np.maximum(deep, support * np.maximum(key, core_w)), 0.0, 1.0)

    # ------------------------------------------------- nhuộm cho MỘT người
    def _recolor_one(self, frame, m, label, kps):
        """Đổi màu vùng tóc `m` (mask riêng của người này) sang màu đích của họ.

        Mọi thống kê (trung bình L/a/b, độ lệch chuẩn) đều tính TRÊN MASK CỦA NGƯỜI NÀY.
        Đó là lý do phải chạy riêng từng người chứ không nhuộm một lượt: tóc A có thể tối hơn
        tóc B, mà công thức dịch độ sáng lấy trung bình làm mốc - trộn hai người vào một mốc
        thì cả hai đều lệch.
        """
        H, W = frame.shape[:2]
        Lt, At, Bt = self.target_lab[label]

        ys, xs = np.nonzero(m > 0.02)
        if len(ys) < 20:
            return False

        sx, sy = W / SEG_SIZE, H / SEG_SIZE
        mx1, mx2 = int(xs.min()), int(xs.max()) + 1
        my1, my2 = int(ys.min()), int(ys.max()) + 1

        if HAIR_WISPS > 0 and kps is not None:
            # Nới hộp ra bao cả khuôn mặt: tóc mai rủ TRƯỚC MẶT, nằm ngoài hộp bao của mái
            # tóc, nên không nới thì tầng tóc mai không có gì để nhìn.
            k256 = np.asarray(kps, dtype=np.float64) / np.array([sx, sy])
            d256 = float(np.linalg.norm(k256[1] - k256[0]))
            cx, cy = k256.mean(0)
            mx1 = max(0, min(mx1, int(np.floor(cx - 2.2 * d256))))
            mx2 = min(SEG_SIZE, max(mx2, int(np.ceil(cx + 2.2 * d256))))
            my1 = max(0, min(my1, int(np.floor(cy - 2.0 * d256))))
            my2 = min(SEG_SIZE, max(my2, int(np.ceil(cy + 3.0 * d256))))

        x1 = max(0, int(np.floor(mx1 * sx))); x2 = min(W, int(np.ceil(mx2 * sx)))
        y1 = max(0, int(np.floor(my1 * sy))); y2 = min(H, int(np.ceil(my2 * sy)))
        if x2 - x1 < 4 or y2 - y1 < 4:
            return False

        crop = frame[y1:y2, x1:x2]
        sub = m[my1:my2, mx1:mx2]
        cov = np.clip(cv2.resize(sub, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR), 0.0, 1.0)

        lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
        L, A, B = lab[..., 0], lab[..., 1], lab[..., 2]

        if HAIR_EDGE_SMART:
            edge_px = max(3, int(round(3.0 * (H + W) / 2.0 / SEG_SIZE)))
            cov = self._edge_matte(L, cov, edge_px)

        if HAIR_WISPS > 0 and kps is not None:
            kl = np.asarray(kps, dtype=np.float64) - np.array([x1, y1], dtype=np.float64)
            d_eye = float(np.linalg.norm(kl[1] - kl[0]))
            core = cov >= 0.9
            if d_eye > 8 and core.sum() >= 50:
                wisp = self._wisp_matte(L, cov, kl, d_eye, float(L[core].mean()))
                cov = np.maximum(cov, HAIR_WISPS * wisp)

        # Giữ lại để cell 6.1 vẽ ra ĐÚNG vùng đã nhuộm của từng người.
        self.last_alpha[label], self.last_box[label] = cov, (x1, y1, x2, y2)

        w = cov * HAIR_STRENGTH
        wsum = float(w.sum())
        if wsum < 20:
            return False

        mL = float((L * w).sum() / wsum)
        mA = float((A * w).sum() / wsum)
        mB = float((B * w).sum() / wsum)
        sL = float(np.sqrt(max(((L - mL) ** 2 * w).sum() / wsum, 1e-6)))

        # --- ĐỘ SÁNG: dịch cả phân bố, giữ biến thiên, KHUẾCH ĐẠI phần sợi mảnh ---
        # (L - mL) là sợi tóc / bóng đổ / nếp tóc. Nhưng giữ nguyên là chưa đủ khi nhuộm tóc
        # tối sang màu sáng: tóc đen sd 9-12, tóc vàng thật 20-35. Dịch mà không khuếch đại
        # thì ra mảng sáng đều -> cảm giác "quét sơn". Chỉ khuếch đại PHẦN CHI TIẾT, không
        # đụng bóng đổ lớn (nhân cả bóng đổ thì tóc lồi lõm loang lổ).
        sigma_d = float(np.clip(0.010 * max(crop.shape[0], crop.shape[1]), 1.5, 6.0))
        base = cv2.GaussianBlur(L, (0, 0), sigmaX=sigma_d)
        detail = L - base
        auto = float(np.clip(abs(Lt - mL) / 90.0, 0.0, 1.6))
        gain = 1.0 + auto * HAIR_DETAIL
        L_new = (mL + (base - mL) * HAIR_CONTRAST + (Lt - mL) * HAIR_LIGHTNESS
                 + detail * gain)

        # Nén mềm hai đầu thay vì để np.clip cắt cụt ở 0/255: cắt cụt làm mọi sợi sáng dính
        # bết thành một mảng phẳng. tanh có đạo hàm 1 tại điểm gấp nên nối liền mượt.
        knee_hi, knee_lo = 205.0, 45.0
        over = L_new > knee_hi
        if over.any():
            L_new[over] = knee_hi + (255.0 - knee_hi) * np.tanh(
                (L_new[over] - knee_hi) / (255.0 - knee_hi))
        under = L_new < knee_lo
        if under.any():
            L_new[under] = knee_lo - knee_lo * np.tanh((knee_lo - L_new[under]) / knee_lo)

        # --- MÀU: thay a/b bằng màu đích, giữ lại một phần biến thiên gốc ---
        A_new = At + (A - mA) * HAIR_KEEP_TONE
        B_new = Bt + (B - mB) * HAIR_KEEP_TONE
        if HAIR_SATURATION != 1.0:
            A_new = 128.0 + (A_new - 128.0) * HAIR_SATURATION
            B_new = 128.0 + (B_new - 128.0) * HAIR_SATURATION

        # --- Chừa vùng bắt sáng: phản chiếu trên tóc thật gần như không màu ---
        # Ngưỡng theo phân bố độ sáng CỦA CHÍNH MÁI TÓC NÀY (mL, sL), không phải hằng số.
        wc = w
        if HAIR_KEEP_HIGHLIGHT > 0:
            hl = np.clip((L - (mL + 1.5 * sL)) / max(1.5 * sL, 1e-6), 0.0, 1.0)
            wc = w * (1.0 - hl * HAIR_KEEP_HIGHLIGHT)

        lab[..., 0] = L + (L_new - L) * w        # độ sáng: nhuộm cả vùng bóng
        lab[..., 1] = A + (A_new - A) * wc       # màu: chừa vùng bóng lại
        lab[..., 2] = B + (B_new - B) * wc

        np.clip(lab, 0, 255, out=lab)
        frame[y1:y2, x1:x2] = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
        self.n_applied[label] = self.n_applied.get(label, 0) + 1
        return True

    # ---------------------------------------------------------------- nhuộm
    def apply(self, frame, people):
        """Nhuộm tóc cho những người trong `people`. Ghi trực tiếp vào frame.

        people = {label: {'center': (x, y), 'width': w, 'kps': 5x2 hoặc None}}
          center/width : tâm và bề ngang khuôn mặt, dùng để CHIA mask tóc.
          kps          : chỉ cần cho tầng tóc mai; None thì bỏ qua tầng đó.

        Trả (frame, danh sách label đã nhuộm được).
        """
        self.last_alpha, self.last_box = {}, {}
        # Chỉ xét người vừa có vị trí, vừa có màu đích. Người đặt màu None thì không nhuộm,
        # NHƯNG vẫn phải góp mặt vào phép chia - nếu bỏ họ ra thì tóc của họ bị coi là "gần
        # người kia nhất" và ăn màu của người kia.
        placed = {l: i for l, i in people.items() if i is not None and i.get('center') is not None}
        if not placed:
            return frame, []
        wanted = [l for l in placed if self.target_lab.get(l) is not None]
        if not wanted:
            return frame, []

        H, W = frame.shape[:2]
        sx, sy = W / SEG_SIZE, H / SEG_SIZE

        hair, face = self._segment(frame)
        if self.prev_masks is not None and HAIR_MASK_SMOOTH > 0:
            hair = HAIR_MASK_SMOOTH * self.prev_masks[0] + (1 - HAIR_MASK_SMOOTH) * hair
            face = HAIR_MASK_SMOOTH * self.prev_masks[1] + (1 - HAIR_MASK_SMOOTH) * face
        self.prev_masks = (hair, face)

        lo, hi = HAIR_CONF
        m = np.clip((hair - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
        r = int(round(HAIR_EDGE_CHOKE))
        if r >= 1:
            m = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
        if HAIR_FEATHER > 0:
            m = cv2.GaussianBlur(m, (0, 0), sigmaX=HAIR_FEATHER)
        if HAIR_PROTECT_FACE:
            # Trừ da mặt SAU khi làm mềm mép, không phải trước: làm mềm trước rồi trừ sau thì
            # chính cái blur lại kéo mask tràn ngược vào trán vài pixel -> da bị ám màu ở
            # chân tóc. Mask da mặt của MediaPipe vốn đã mềm sẵn.
            m *= (1.0 - face)

        parts = self._split_by_person(m, placed, sx, sy)

        done = []
        for label in wanted:
            if self._recolor_one(frame, parts[label], label, placed[label].get('kps')):
                done.append(label)
        return frame, done


recolor = None

if not USE_HAIR:
    print('USE_HAIR = False -> giữ nguyên màu tóc của cả hai người.')
elif not HAIR_AVAILABLE:
    print('USE_HAIR = True nhưng mediapipe không dùng được -> chạy tiếp, chỉ swap mặt.')
else:
    recolor = HairRecolor(SEGMENTER_PATH)
    _src_img = {'A': source_img_a, 'B': source_img_b}

    for _label, _want in (('A', HAIR_COLOR_A), ('B', HAIR_COLOR_B)):
        if _want is None:
            recolor.set_target(_label, None)
            print(f'{_label}: không đổi màu tóc (giữ nguyên tóc gốc trong video).')
            continue
        if _want == 'ảnh':
            _bgr, _cover = recolor.hair_color_of(_src_img[_label])
            print(f'{_label}: màu tóc lấy từ ảnh nguồn {_label} = {bgr_to_hex(_bgr)}  '
                  f'(vùng tóc chiếm {_cover:.1%} ảnh)')
            if _cover < 0.02:
                print(f'   (vùng tóc trong ảnh {_label} khá nhỏ -> màu lấy được có thể '
                      f'không đại diện)')
        else:
            _bgr = hex_to_bgr(HAIR_PALETTE.get(_want, _want))[0, 0]
            print(f'{_label}: {_want} = {bgr_to_hex(_bgr)}')
        recolor.set_target(_label, _bgr)
        _L, _A, _B = recolor.target_lab[_label]
        print(f'   LAB đích: L={_L:.0f}/255  a={_A - 128:+.0f}  b={_B - 128:+.0f}')

    if recolor.target_lab:
        print('Sẵn sàng nhuộm tóc.')
    else:
        print('Không ai được đặt màu -> phần nhuộm sẽ không làm gì.')
        recolor = None

In [ ]:
# Xem màu tóc đã chọn + mask tóc trên hai ảnh nguồn, TRƯỚC khi chạy cả video.
import numpy as np
import cv2

if recolor is None:
    print('Không có recolor -> không có gì để xem.')
else:
    from google.colab.patches import cv2_imshow

    # ---- 1. mask tóc trên từng ảnh nguồn (kiểm tra segment có ăn không) ----
    # Ảnh nguồn chỉ có MỘT người nên ở đây không cần chia mask; đây là chỗ kiểm tra
    # MediaPipe có nhận ra tóc trong ảnh của bạn hay không.
    for label, img in (('A', source_img_a), ('B', source_img_b)):
        h, w = img.shape[:2]
        hair, face = recolor._segment(img)
        lo, hi = HAIR_CONF
        m = np.clip((hair - lo) / max(hi - lo, 1e-6), 0.0, 1.0)   # cùng thứ tự với apply():
        if HAIR_FEATHER > 0:                                      # ngưỡng -> làm mềm -> trừ da
            m = cv2.GaussianBlur(m, (0, 0), sigmaX=HAIR_FEATHER)
        if HAIR_PROTECT_FACE:
            m *= (1.0 - face)
        m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)[..., None]

        red = np.zeros_like(img); red[..., 2] = 255
        overlay = (img * (1 - 0.55 * m) + red * (0.55 * m)).astype(np.uint8)
        strip = np.hstack([img, overlay])
        sc = min(1.0, 900 / strip.shape[1])
        if sc < 1.0:
            strip = cv2.resize(strip, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
        print(f'Ảnh nguồn {label} — mask tóc (đỏ = MediaPipe nhận là tóc):')
        cv2_imshow(strip)
    print('  lem sang nền/da -> nâng HAIR_CONF, vd (0.5, 0.8)')
    print('  hụt rìa tóc     -> hạ HAIR_CONF, vd (0.25, 0.5)')
    print()

    # ---- 2. màu đích của A và B, cạnh nhau ----
    sw = np.full((110, 640, 3), 255, np.uint8)
    for i, label in enumerate(LABELS):
        x0 = i * 320
        bgr = recolor.target_bgr.get(label)
        if bgr is None:
            cv2.rectangle(sw, (x0 + 8, 8), (x0 + 312, 100), (230, 230, 230), -1)
            cv2.putText(sw, f'{label}: no change', (x0 + 20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (90, 90, 90), 2, cv2.LINE_AA)
        else:
            cv2.rectangle(sw, (x0 + 8, 8), (x0 + 312, 100), [int(v) for v in bgr], -1)
            # cv2.putText không vẽ được dấu tiếng Việt -> chỉ ghi nhãn + mã hex.
            cv2.putText(sw, f'{label}  {bgr_to_hex(bgr)}', (x0 + 20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    print('Màu đích: A (trái)  |  B (phải)')
    cv2_imshow(sw)

    # ---- 3. bảng màu để tra tên ----
    names = list(HAIR_PALETTE)
    cols, cw, ch = 6, 150, 70
    rows = (len(names) + cols - 1) // cols
    board = np.full((rows * ch, cols * cw, 3), 255, np.uint8)
    for i, nm in enumerate(names):
        r, c = divmod(i, cols)
        y, x = r * ch, c * cw
        cv2.rectangle(board, (x + 4, y + 4), (x + cw - 4, y + ch - 22),
                      [int(v) for v in hex_to_bgr(HAIR_PALETTE[nm])[0, 0]], -1)
        if nm in (HAIR_COLOR_A, HAIR_COLOR_B):
            who = '+'.join(l for l, v in (('A', HAIR_COLOR_A), ('B', HAIR_COLOR_B)) if v == nm)
            cv2.rectangle(board, (x + 2, y + 2), (x + cw - 2, y + ch - 20), (0, 0, 255), 2)
            cv2.putText(board, who, (x + cw - 30, y + ch - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 200), 2, cv2.LINE_AA)
        cv2.putText(board, str(i), (x + 8, y + ch - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    print()
    print('Bảng màu (viền đỏ = đang chọn, kèm tên người):')
    cv2_imshow(board)
    print('  ' + ' | '.join(f'{i}={nm}' for i, nm in enumerate(names)))

## 6. Xử lý video

Mục này chia làm nhiều cell nhỏ, chạy tuần tự:

| Cell | Làm gì |
|---|---|
| **5b** (ở trên) | `HairRecolor` — chia mask tóc A/B và nhuộm từng người |
| **6.1** | `FaceTracker` — mặt nào là A, mặt nào là B |
| **6.1b** | `swap_all_faces` — swap cả hai cùng lúc, không ai đè lên ai |
| **6.2 / 6.2b** | Chẩn đoán: đo số liệu thật từ video (chỉ đọc, không tạo video) |
| **6.3** | Định nghĩa `process_frame()` |
| **6.4** | Thử **một frame** — xem swap đúng người chưa, tóc chia đúng chưa |
| **6.5** | Thử **nhiều cặp màu** trên một frame |
| **6.6** | Chạy **toàn bộ video** |

Mỗi frame đi qua bốn bước, theo đúng thứ tự này:

1. `app.get(frame)` → detect mọi mặt; `tracker.step()` → gán A/B theo **embedding ArcFace**
2. `swap_all_faces(frame, ...)` → swap **từ frame GỐC**, tranh chấp pixel được phân xử
3. `enhance_face_region(...)` → làm nét từng mặt, chỉ khi `restorer is not None`
4. `recolor.apply(...)` → **chia mask tóc rồi nhuộm từng người**, chỉ khi `recolor is not None`

**Bước 4 phải nằm sau bước 3.** Ô crop của GFPGAN nới bbox thêm 40% nên lấn sang cả tóc; nhuộm
trước thì GFPGAN vẽ lại chính vùng tóc vừa nhuộm, mỗi frame một kiểu → nhấp nháy.

> **Bước 1-3 cần gán được danh tính, bước 4 chỉ cần biết mặt Ở ĐÂU.** Frame nào tracker không
> gán được ai thì mặt giữ nguyên, nhưng tóc vẫn nhuộm được bằng **vị trí lần thấy cuối**.
> Dòng thống kê cuối mục 6.6 in riêng số frame phải dùng vị trí cũ cho từng người.

### Vì sao phải khớp theo DANH TÍNH, không phải theo vị trí

**Vấn đề cần giải quyết:** model detect mặt mỗi frame **không đảm bảo thứ tự cố định** (frame này trả về [trái, phải], frame sau có thể trả về [phải, trái], đặc biệt khi 2 người quay đầu/che nhau lúc hôn). Nếu chỉ lấy theo index `[0]`, `[1]` thì mặt A/B sẽ bị **đảo lộn giữa các frame**, tạo hiệu ứng giật/lóe rất xấu.

**Cách xử lý ở đây — khớp theo *danh tính*, không chỉ theo vị trí:**

1. Ở frame đầu tiên detect được mặt, quy ước **A = mặt bên trái khung hình, B = mặt bên phải**, rồi lưu lại `normed_embedding` (vector nhận dạng ArcFace mà InsightFace **đã tính sẵn** ở bước detect — không tốn thêm chi phí) làm *danh tính tham chiếu* của từng người.
2. Mỗi frame sau, chi phí gán một khuôn mặt cho A/B = `(1 − cosine_similarity với embedding tham chiếu) + 0.25 × (khoảng cách vị trí đã chuẩn hoá)`. Embedding quyết định chính, vị trí chỉ phá hoà.
3. Mặt có `cosine_similarity < 0.20` bị **loại thẳng** — người lạ đi ngang hay người thứ 3 trong khung sẽ không bị swap nhầm (bản cũ luôn lấy mặt "gần nhất" nên không tránh được).
4. Việc gán được giải **tối ưu toàn cục** (duyệt hết các cách ghép, chỉ 2 người nên rất rẻ) thay vì greedy theo thứ tự A rồi B — greedy làm A luôn giành mặt trước, kể cả khi mặt đó rõ ràng là của B.

**Trường hợp 1 mặt bị che khuất (occlusion) hoàn toàn:** mặt hiện ra được gán cho **đúng người theo embedding**, người còn lại giữ nguyên frame gốc ở khung đó (không suy đoán mù).

**Người xuất hiện muộn:** nếu frame đầu chỉ detect được 1 mặt, người thứ hai vẫn được **khởi tạo muộn** ngay khi họ hiện ra ở frame sau.

**Ghi và ghép audio trong cùng một pass:** frame thô được ghi thẳng vào `ffmpeg` qua pipe, xuất ra H.264 kèm audio gốc. Bỏ được file trung gian `mp4v` (vốn làm mất chất lượng một lần trước khi re-encode lần hai) và bỏ luôn một lượt decode+encode toàn bộ video.

### 6.1 Bộ tracker (dùng chung cho cả chẩn đoán và xử lý)

Tách riêng khỏi vòng lặp xử lý để **cell chẩn đoán ở 6.2 chạy đúng logic đang dùng thật**, không phải bản chép lại dễ lệch. Chạy cell này trước, nó không đụng gì tới video nên rất nhanh.

**Ngưỡng tuyệt đối vs. so sánh tương đối.** Số đo thật từ video cho thấy lúc hôn, mặt người B nghiêng mạnh làm `sim` với chính danh tính của cô ấy tụt còn **+0.24**, chỉ cách ngưỡng loại (`MIN_SIM = 0.20`) đúng 0.04 — tụt thêm chút là cô ấy bị bỏ qua, mặt gốc lộ ra, nhấp nháy. Nhưng so sánh hai cách ghép thì lại cực chắc:

```
ghép đúng:  nam→A (+0.80) + nữ→B (+0.24) = 1.04
ghép đảo:   nam→B (−0.06) + nữ→A (+0.17) = 0.11   -> chênh 0.93
```

Giá trị tuyệt đối là thứ suy sụp khi mặt nghiêng; **thứ tự tương đối thì không**. Nên khi số mặt detect được đúng bằng số người đã biết — bài toán chỉ còn là "mặt nào là ai" — ngưỡng tuyệt đối được nới xuống `MIN_SIM_RELAXED` và để phép ghép tối ưu tự quyết. Ngưỡng chặt vẫn giữ nguyên cho trường hợp có mặt thừa, để còn lọc người lạ đi ngang.

In [ ]:
import itertools
import numpy as np

# ---------------- Tham số tracking ----------------
POS_W           = 0.25   # trọng số vị trí; embedding vẫn là yếu tố quyết định
MIN_SIM         = 0.20   # ngưỡng chặt: dùng khi có mặt thừa (lọc người lạ)
MIN_SIM_RELAXED = 0.10   # ngưỡng nới: dùng khi số mặt == số người đã biết.
                         # Không hạ thấp hơn: hai embedding của HAI NGƯỜI KHÁC NHAU
                         # dao động quanh 0 với độ lệch ~1/sqrt(512) = 0.044, nên 0.10
                         # (~2.3 sigma) vẫn chặn được người lạ, trong khi mặt nghiêng
                         # của chính chủ đo được thấp nhất là +0.24 — còn dư biên.
MAX_DIST_RATIO  = 0.15   # bán kính chuẩn hoá khoảng cách, theo cạnh lớn khung hình
EMB_EMA         = 0.05   # tốc độ cập nhật embedding tham chiếu
EMB_EMA_MIN_SIM = 0.50   # chỉ cập nhật khi khớp chắc -> tránh trôi sang nhầm người
UNASSIGNED_COST = 1.5    # phạt khi để một người không được gán

LABELS = ('A', 'B')


def face_center(face):
    x1, y1, x2, y2 = face.bbox
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0])


def face_width(face):
    return float(face.bbox[2] - face.bbox[0])


class FaceTracker:
    """Gán các khuôn mặt detect được ở mỗi frame cho đúng người A / B.

    Khớp theo danh tính (embedding ArcFace InsightFace đã tính sẵn), vị trí chỉ phá hoà.
    Việc gán được giải tối ưu toàn cục thay vì greedy theo thứ tự nhãn.
    """

    def __init__(self, frame_w, frame_h):
        self.max_dist = MAX_DIST_RATIO * max(frame_w, frame_h)
        self.ref_emb = {l: None for l in LABELS}
        self.last_pos = {l: None for l in LABELS}

    def sim_to(self, face, label):
        ref = self.ref_emb[label]
        if ref is None:
            return None
        return float(np.dot(face.normed_embedding, ref))

    def pos_cost(self, center, label):
        if self.last_pos[label] is None:
            return 1.0
        return min(float(np.linalg.norm(center - self.last_pos[label])) / self.max_dist, 1.0)

    def match_cost(self, face, center, label, min_sim=MIN_SIM):
        """Chi phí gán `face` cho `label`; None = không được phép gán."""
        s = self.sim_to(face, label)
        if s is None or s < min_sim:
            return None
        return (1.0 - s) + POS_W * self.pos_cost(center, label)

    def solve(self, labels, faces, centers, min_sim=MIN_SIM):
        """Gán tối ưu toàn cục (brute force — tối đa 2 người nên rất rẻ)."""
        best_total, best_map = None, {}
        idx_choices = [None] + list(range(len(faces)))
        for combo in itertools.product(idx_choices, repeat=len(labels)):
            used = [c for c in combo if c is not None]
            if len(set(used)) != len(used):
                continue                             # một mặt không thể là hai người
            total, mapping = 0.0, {}
            for label, idx in zip(labels, combo):
                if idx is None:
                    total += UNASSIGNED_COST
                    continue
                c = self.match_cost(faces[idx], centers[idx], label, min_sim)
                if c is None:
                    total = None
                    break
                total += c
                mapping[label] = idx
            if total is None:
                continue
            if best_total is None or total < best_total:
                best_total, best_map = total, mapping
        return best_map

    def step(self, faces):
        """Xử lý một frame. Trả về (assigned, debug).

        assigned thiếu nhãn = frame này không gán được người đó (giữ nguyên mặt gốc).
        debug chứa sim/cost của mọi mặt với mọi nhãn, đo TRƯỚC khi cập nhật state.
        """
        centers = [face_center(f) for f in faces]
        known = [l for l in LABELS if self.ref_emb[l] is not None]

        # Số mặt đúng bằng số người đã biết -> bài toán chỉ là "mặt nào là ai", một câu
        # hỏi TƯƠNG ĐỐI. Ngưỡng tuyệt đối lúc này là thứ mong manh nhất (mặt nghiêng làm
        # sim tụt sát ngưỡng dù cách ghép đúng vẫn hơn cách ghép đảo rất xa), nên nới ra.
        relaxed = bool(known) and len(faces) == len(known)
        min_sim = MIN_SIM_RELAXED if relaxed else MIN_SIM

        debug = {
            'centers': centers,
            'sim':  [{l: self.sim_to(f, l) for l in LABELS} for f in faces],
            'cost': [{l: self.match_cost(f, centers[i], l, min_sim) for l in LABELS}
                     for i, f in enumerate(faces)],
            'late_init': [],
            'min_sim': min_sim,
            'relaxed': relaxed,
        }

        assigned = self.solve(known, faces, centers, min_sim) if known else {}

        # Khởi tạo muộn: người xuất hiện sau frame đầu vẫn phải được nhận diện.
        unknown = [l for l in LABELS if self.ref_emb[l] is None]
        if unknown:
            taken = set(assigned.values())
            free = sorted((i for i in range(len(faces)) if i not in taken),
                          key=lambda i: centers[i][0])
            for label in unknown:                    # quy ước: A = bên trái, B = bên phải
                if not free:
                    break
                i = free.pop(0) if label == 'A' else free.pop(-1)
                assigned[label] = i
                self.ref_emb[label] = faces[i].normed_embedding.copy()
                debug['late_init'].append(label)

        for label, idx in assigned.items():
            self.last_pos[label] = centers[idx]
            # Cập nhật nhẹ embedding tham chiếu để bám theo góc mặt/ánh sáng,
            # nhưng chỉ khi khớp chắc chắn -> tránh trôi dần sang nhầm người.
            s = self.sim_to(faces[idx], label)
            if s is not None and s >= EMB_EMA_MIN_SIM:
                e = (1 - EMB_EMA) * self.ref_emb[label] + EMB_EMA * faces[idx].normed_embedding
                self.ref_emb[label] = e / (np.linalg.norm(e) + 1e-8)

        return assigned, debug


print(f'FaceTracker sẵn sàng.  POS_W={POS_W}  MIN_SIM={MIN_SIM} '
      f'(nới còn {MIN_SIM_RELAXED} khi số mặt khớp số người)')

### 6.1b Ghép mặt đã swap mà không để ai đè lên ai

Chỗ này quyết định pixel nào thuộc về ai. Bốn lớp:

**1. Không dùng mask mặc định của inswapper.** Nó dán về bằng cả ô vuông 128×128 đã align (`img_mask = img_white` trong source; mask tinh `fake_diff` được tính nhưng dòng dùng bị comment), rộng gấp ~1.5–1.7 lần bề ngang mặt → lúc hôn thì trùm sang mặt người kia. Ở đây tự ghép, giới hạn trong ellipse bám khuôn mặt.

**2. Mask phân vùng khuôn mặt** (`facexlib` parsenet, đã có sẵn). Loại nền, tóc, cổ, quần áo khỏi vùng dán.

**3. Nhường ở mức BỘ PHẬN, không phải mức pixel.** Đây là phần mới.

Nhường theo pixel làm ranh giới rơi vào chỗ hai mask cân bằng — một đường tuỳ tiện **có thể cắt ngang giữa cái mũi hoặc giữa cái môi**. Nửa cái mũi được swap, nửa kia không: mắt người đọc ra ngay là vỡ. Nhường theo bộ phận thì ranh giới chạy theo **đường viền giải phẫu**, thứ mắt người vốn quen nhìn.

Xác định bộ phận thuộc về ai bằng **vị trí giải phẫu**: crop đã được align theo template arcface, nên mũi của chính khuôn mặt đó **phải** nằm gần vị trí mũi chuẩn (x≈64, y≈72 trong crop 128). Mũi của người bên cạnh lọt vào crop sẽ nằm lệch hẳn. Với mỗi nhãn bộ phận, lấy các thành phần liên thông, giữ thành phần gần vị trí chuẩn nhất; các thành phần còn lại là **của người khác** và bị trừ khỏi vùng dán.

Nhờ vậy mũi A vẫn được swap (nó là của A), chỉ mask của B phải chừa chỗ đó ra. Nếu không thành phần nào đủ gần vị trí chuẩn thì bộ phận đó thành "của không ai" → bị loại khỏi cả hai → giữ pixel gốc.

**Giới hạn thật:** cách này chỉ tách được các bộ phận **rời nhau** (mũi, môi, mắt, chân mày). Vùng `skin` của hai người khi áp má thì **liền một khối**, thành phần liên thông không tách được — phần đó vẫn phải dựa vào ellipse và luật ở lớp 4.

**4. Vùng còn tranh chấp** — chủ yếu là da má chỗ tiếp xúc:

- chỉ một người đòi → người đó lấy
- cả hai đòi, một bên là bộ phận nhô ra của chính mình (mũi/môi) → bên đó nằm trước
- cả hai đòi ngang nhau → **không dán ai cả, giữ pixel gốc**

**5. Đang chạm nhau thì chừa hẳn mũi + miệng + môi ra** (`NO_SWAP_MOUTH_NOSE`).

Mỗi frame tính `khoảng cách 2 tâm mặt / bề ngang mặt`; dưới `CONTACT_RATIO` thì coi là đang chạm và trừ vùng mũi/môi của **cả hai** khỏi mask, giữ nguyên pixel gốc. Có điều kiện, nên frame nào hai người tách ra vẫn swap đầy đủ.

Vùng mũi/môi được dựng từ **template arcface** chứ không trông chờ parsenet. Lý do: số đo thật từ video cho thấy khi mặt nghiêng, parsenet trả về diện tích mũi/môi chỉ **0–115px** trong khi vùng thật khoảng **2000px**, và ở 5 frame chồng lấn nặng nhất thì ra **0 tuyệt đối** — trừ đi số 0 thì không chừa được gì. Crop đã align nên vị trí mũi/miệng là cố định, dựng hình học luôn chắc chắn hơn. parsenet vẫn được hợp nhất vào khi nó nhận ra, để thêm độ chính xác.

Các cờ tắt/bật: `USE_PART_OWNERSHIP` (lớp 3), `NO_SWAP_MOUTH_NOSE` (lớp 5), `MOUTH_NOSE_FROM_TEMPLATE` (nguồn dựng vùng mũi/môi).

In [ ]:
import numpy as np
import cv2

# ---------------- Cấu hình ghép ----------------
USE_PARSING         = True    # mask phân vùng khuôn mặt (facexlib parsenet)
USE_PART_OWNERSHIP  = True    # nhường ở mức bộ phận thay vì mức pixel
PROTRUDE_BONUS      = 1.0     # điểm cộng cho bộ phận nhô ra của chính mình khi tranh chấp
CONFLICT_BAND       = 0.35    # dưới mức chênh này coi như tranh chấp -> nhường pixel gốc

# Khi hai khuôn mặt đang chạm nhau thì CHỪA HẲN mũi + miệng + môi ra, giữ nguyên pixel gốc.
# Đây là vùng biến dạng nặng nhất lúc hôn. Đánh đổi: mũi và môi mang danh tính rất nặng,
# bỏ chúng đi thì kết quả bớt giống người nguồn — đặt False để so hai bản trên cùng video.
NO_SWAP_MOUTH_NOSE  = True
CONTACT_RATIO       = 1.00    # khoảng cách 2 tâm mặt / bề ngang mặt; nhỏ hơn = đang chạm.
                              # Số đo thật từ video: min 0.78, trung vị 0.88 -> 40/56 frame
                              # dưới ngưỡng, tức ngưỡng này kích hoạt đúng.

# Vùng mũi/môi dựng từ TEMPLATE thay vì trông chờ parsenet nhận ra.
# Số đo thật: khi mặt nghiêng, parsenet cho diện tích mũi/môi chỉ 0-115px trong khi
# vùng thật khoảng 2000px — ở 5 frame chồng lấn nặng nhất thì ra 0 tuyệt đối. Trừ đi
# số 0 thì không chừa được gì, nên không thể chỉ dựa vào nó.
MOUTH_NOSE_FROM_TEMPLATE = True
MOUTH_NOSE_W        = 0.55    # bán trục ngang, tính theo khoảng cách 2 mắt trong crop
MOUTH_NOSE_H        = 0.60    # bán trục dọc

# Mặt nghiêng sâu quá thì MỜ DẦN rồi thôi không dán.
# inswapper_128 huấn luyện chủ yếu trên mặt gần chính diện, nghiêng mạnh thì nó
# "làm phẳng" khuôn mặt về hướng chính diện -> dán lên đầu đang nghiêng nhìn ra ngay
# là giả. Qua một mức nào đó, giữ mặt gốc trông tự nhiên hơn là dán.
# Dùng dải MỜ DẦN chứ không cắt phựt, để không bị giật khi frame vượt ngưỡng.
# MẶC ĐỊNH TẮT. Số đo thật từ video cho thấy với ngưỡng ban đầu, lớp này tắt swap trên
# 32/56 frame của A và 44/56 của B — phá video chứ không cứu. Bật lên sau khi đã xem
# phân bố yaw thật ở cell chẩn đoán và chọn ngưỡng theo số đo.
YAW_FADE_ON         = False
YAW_FULL            = 0.35    # dưới mức này: dán đầy đủ  (thang [0,1])
YAW_NONE            = 0.75    # trên mức này: không dán nữa

# Tỉ lệ (khoảng cách 2 mắt) / (mắt->miệng) của mặt chính diện chuẩn, tính từ template
# arcface: 35.2377 / 40.6866 = 0.86608. Dùng làm mốc để quy độ nghiêng về thang [0,1].
FRONTAL_IOD_RATIO   = 0.86608
FOREIGN_DILATE      = 0.02    # nới vùng bộ phận của người khác, theo cạnh crop
PART_MAX_DIST       = 0.25    # bộ phận lệch quá bao nhiêu (theo cạnh crop) thì coi là của người khác
PARSE_SIZE          = 512
SWAP_SIZE           = 128

# Nhãn parsenet (CelebAMask-HQ 19 lớp). MASK_COLORMAP lấy nguyên từ facexlib:
# giữ da + mắt + mũi + môi + tai, bỏ nền(0) / cổ(14) / quần áo(16) / tóc(17) / mũ(18).
_MASK_COLORMAP = [0, 255, 255, 255, 255, 255, 255, 255, 255, 255,
                  255, 255, 255, 255, 0, 255, 0, 0, 0]
FACE_LABELS     = [i for i, c in enumerate(_MASK_COLORMAP) if c == 255]
PROTRUDE_LABELS = [10, 11, 12, 13]        # nose, mouth, upper lip, lower lip
HAIR_LABELS     = [17, 18]                # hair, hat

# inswapper SINH RA cả ô 128x128, trong đó có tóc của người nguồn ở rìa. Nếu vùng tóc
# lọt vào mask thì tóc lạ bị dán vào video. Không thể chỉ trông vào FACE_LABELS: ở mặt
# nghiêng parsenet nhận tóc chập chờn, và ellipse dựng theo tỉ lệ template thì vươn ra
# quá mép mặt thật (mặt nghiêng hẹp hơn template giả định).
HAIR_DILATE     = 0.03    # nới vùng tóc trước khi trừ, theo cạnh crop
MASK_ERODE      = 0.02    # co viền mask vào trong — chặn rò rỉ kể cả khi parsenet
                          # không nhận ra tóc. Đặt 0 nếu thấy mất nhiều phần mặt.

# Bộ phận nào neo vào mốc giải phẫu nào. Chỉ liệt kê các bộ phận RỜI NHAU —
# `skin` của hai người khi áp má là một khối liền, không tách được kiểu này.
PART_ANCHOR = {10: 'nose', 11: 'mouth', 12: 'mouth', 13: 'mouth',
               4: 'l_eye', 5: 'r_eye', 2: 'l_eye', 3: 'r_eye'}

_PARSER = None
_PARSE_DEVICE = None
_WARNED = set()


def _warn_once(key, msg):
    """In cảnh báo một lần thôi — vòng lặp chạy hàng trăm frame."""
    if key not in _WARNED:
        _WARNED.add(key)
        print(msg)


def canon_points(size=SWAP_SIZE):
    """Vị trí giải phẫu chuẩn trong crop đã align, suy thẳng từ template arcface.

    estimate_norm với image_size chia hết cho 128 dùng ratio = size/128 và dịch x thêm
    8*ratio. Với crop 128: hai mắt (46.3, 51.7) và (81.5, 51.5), mũi (64.0, 71.7),
    hai khoé miệng (49.6, 92.4) và (78.7, 92.2).
    """
    s = size / 128.0
    return {
        'l_eye': (46.29 * s, 51.70 * s),
        'r_eye': (81.53 * s, 51.50 * s),
        'nose':  (64.03 * s, 71.74 * s),
        'mouth': (64.14 * s, 92.28 * s),
    }


def split_own_foreign(labels, size):
    """Tách bộ phận CỦA khuôn mặt ở giữa crop khỏi bộ phận của người bên cạnh.

    Crop đã align nên mỗi bộ phận phải nằm gần vị trí chuẩn của nó. Với mỗi nhãn,
    lấy các thành phần liên thông: thành phần gần mốc chuẩn nhất (và đủ gần) là của
    khuôn mặt này; các thành phần khác là của người bên cạnh lọt vào crop.

    Trả về (own_protrude, foreign) — own_protrude chỉ gồm mũi/môi của chính mình,
    dùng để phá hoà ở bước sau; foreign là vùng phải trừ khỏi mask.
    """
    canon = canon_points(size)
    max_d = PART_MAX_DIST * size
    min_area = max(int((size / 32.0) ** 2), 8)      # bỏ đốm nhiễu li ti
    own_prot = np.zeros((size, size), np.float32)
    foreign = np.zeros((size, size), np.float32)

    for lab, anchor in PART_ANCHOR.items():
        blob = (labels == lab).astype(np.uint8)
        if not blob.any():
            continue
        n, cc, stats, cents = cv2.connectedComponentsWithStats(blob, 8)
        ax, ay = canon[anchor]

        best, best_d = None, None
        for k in range(1, n):
            if stats[k, cv2.CC_STAT_AREA] < min_area:
                continue
            d = float(np.hypot(cents[k][0] - ax, cents[k][1] - ay))
            if best_d is None or d < best_d:
                best, best_d = k, d

        for k in range(1, n):
            if stats[k, cv2.CC_STAT_AREA] < min_area:
                continue
            comp = (cc == k)
            if k == best and best_d is not None and best_d <= max_d:
                if lab in PROTRUDE_LABELS:
                    own_prot[comp] = 1.0
            else:
                # Không đủ gần mốc chuẩn -> của người bên cạnh (hoặc không của ai).
                foreign[comp] = 1.0

    if foreign.any():
        k = max(int(FOREIGN_DILATE * size), 1)
        foreign = cv2.dilate(foreign, np.ones((k, k), np.uint8), iterations=1)
    return own_prot, foreign


def yaw_score(face):
    """Độ quay ngang của đầu, suy từ 5 điểm mốc. 0 = chính diện, 1 = nghiêng hết cỡ.

    Quay ngang làm khoảng cách hai mắt CO LẠI (phối cảnh), trong khi khoảng cách
    mắt->miệng gần như không đổi. Lấy tỉ số hai đại lượng đó rồi quy về thang [0,1]
    theo mốc chính diện của template arcface.

    Cách cũ (chiếu mũi lên đường nối hai mắt, chia cho khoảng cách hai mắt) đã bị bỏ:
    mẫu số co về 0 khi nghiêng sâu nên giá trị nổ tung — đo trên video thật ra tới
    12.78, khiến ngưỡng không thể đặt được. Cách này bị chặn nên đặt ngưỡng mới có nghĩa.

    Vẫn BẤT BIẾN với nghiêng đầu (roll), kích thước và vị trí — cả hai đại lượng đều
    là độ dài. insightface 0.7.3 không có face.pose (Face.__getattr__ trả None âm thầm)
    nên phải tự tính.
    """
    kps = np.asarray(face.kps, dtype=np.float64)
    vertical = float(np.linalg.norm((kps[0] + kps[1]) / 2.0 - (kps[3] + kps[4]) / 2.0))
    if vertical < 1e-6:
        return 0.0
    iod = float(np.linalg.norm(kps[1] - kps[0]))
    return float(np.clip(1.0 - (iod / vertical) / FRONTAL_IOD_RATIO, 0.0, 1.0))


def yaw_weight(score):
    """Độ đậm của lớp dán theo độ nghiêng: 1 = dán đầy đủ, 0 = không dán."""
    if not YAW_FADE_ON or YAW_NONE <= YAW_FULL:
        return 1.0
    return float(np.clip((YAW_NONE - score) / (YAW_NONE - YAW_FULL), 0.0, 1.0))


def mouth_nose_region(size=SWAP_SIZE):
    """Vùng mũi + miệng + môi, suy thẳng từ template arcface. Luôn có, không phụ
    thuộc parsenet có nhận ra hay không.

    Crop đã align nên các mốc này cố định: mũi (64.0, 71.7), miệng (64.1, 92.3),
    hai mắt cách nhau 35.2px (với crop 128). Ellipse bao từ sống mũi xuống dưới môi.
    """
    cp = canon_points(size)
    iod = cp['r_eye'][0] - cp['l_eye'][0]
    cx = (cp['nose'][0] + cp['mouth'][0]) / 2.0
    cy = (cp['nose'][1] + cp['mouth'][1]) / 2.0
    m = np.zeros((size, size), np.float32)
    cv2.ellipse(m, (int(round(cx)), int(round(cy))),
                (max(int(MOUTH_NOSE_W * iod), 1), max(int(MOUTH_NOSE_H * iod), 1)),
                0, 0, 360, 1.0, -1)
    return m


def build_face_mask(size, scale=1.0):
    """Ellipse bám khuôn mặt, trong không gian crop đã align."""
    m = np.zeros((size, size), np.float32)
    ax = min(int(size * 0.33 * scale), size // 2 - 1)
    ay = min(int(size * 0.47 * scale), size // 2 - 1)
    cv2.ellipse(m, (int(size * 0.50), int(size * 0.53)), (ax, ay), 0, 0, 360, 1.0, -1)
    return m


def init_parser():
    global _PARSER, _PARSE_DEVICE
    if _PARSER is not None:
        return _PARSER
    import torch
    from facexlib.parsing import init_parsing_model
    _PARSE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    _PARSER = init_parsing_model(model_name='parsenet', device=_PARSE_DEVICE)
    _PARSER.eval()
    print(f'  parsenet đã nạp trên {_PARSE_DEVICE}')
    return _PARSER


def parse_crop(aimg):
    """Phân vùng crop đã align, trả về BẢN ĐỒ NHÃN (19 lớp) cùng cỡ với aimg.

    Tiền xử lý chép theo facexlib FaceRestoreHelper để khớp lúc model được train.
    """
    import torch
    parser = init_parser()
    x = cv2.resize(aimg, (PARSE_SIZE, PARSE_SIZE), interpolation=cv2.INTER_LINEAR)
    x = x.astype(np.float32)[:, :, ::-1] / 255.0            # BGR -> RGB, [0,1]
    x = torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1)
    x = ((x - 0.5) / 0.5).unsqueeze(0).to(_PARSE_DEVICE)     # normalize(0.5, 0.5)
    with torch.no_grad():
        labels = parser(x)[0].argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)

    t = max(PARSE_SIZE // 50, 4)                             # bỏ viền đen do warp
    labels[:t, :] = labels[-t:, :] = 0
    labels[:, :t] = labels[:, -t:] = 0
    size = aimg.shape[0]
    return cv2.resize(labels, (size, size), interpolation=cv2.INTER_NEAREST)


def build_masks_for(frame, face, M, size=SWAP_SIZE):
    """Mask vùng dán cho một khuôn mặt. Trả về (mask, mask_nhô_ra_của_mình, tên_nguồn).

    Luôn cắt crop từ `frame` GỐC để mask phản ánh đúng cảnh thật ai đang che ai.
    """
    ell = build_face_mask(size)
    # Vùng mũi/môi hình học luôn dựng được; parsenet chỉ bổ sung độ chính xác khi nó
    # thật sự nhận ra. Hợp nhất hai nguồn để tính năng chừa mũi/môi không phụ thuộc
    # vào việc parsenet có hoạt động ở góc mặt đó hay không.
    base_prot = mouth_nose_region(size) if MOUTH_NOSE_FROM_TEMPLATE         else np.zeros((size, size), np.float32)

    if not USE_PARSING:
        return ell, base_prot, 'ellipse'

    try:
        aimg = cv2.warpAffine(frame, M, (size, size), borderValue=0.0)
        labels = parse_crop(aimg)
    except Exception as e:
        _warn_once('parse', f'  parsenet không dùng được ({type(e).__name__}: {e}) '
                            f'-> dùng ellipse')
        return ell, base_prot, 'ellipse'

    mask = ell * np.isin(labels, FACE_LABELS).astype(np.float32)

    # Trừ tóc CÓ NỚI RỘNG: chỉ loại theo nhãn là chưa đủ, vì viền tóc-da hay bị gán
    # nhầm thành da, mà đúng chỗ đó là nơi tóc người nguồn rò vào.
    hair = np.isin(labels, HAIR_LABELS).astype(np.float32)
    if hair.any() and HAIR_DILATE > 0:
        k = max(int(HAIR_DILATE * size), 1)
        hair = cv2.dilate(hair, np.ones((k, k), np.uint8), iterations=1)
    mask = mask * (1.0 - hair)

    # Co viền mask: hoạt động kể cả khi parsenet KHÔNG nhận ra tóc — trường hợp
    # đã thấy ở mặt nghiêng. Đổi lại thì mất một dải mỏng ở rìa khuôn mặt.
    if MASK_ERODE > 0:
        k = max(int(MASK_ERODE * size), 1)
        mask = cv2.erode(mask, np.ones((k, k), np.uint8), iterations=1)

    if not USE_PART_OWNERSHIP:
        prot = np.maximum(base_prot,
                          np.isin(labels, PROTRUDE_LABELS).astype(np.float32))
        return mask, prot, 'parsenet'

    own_prot, foreign = split_own_foreign(labels, size)
    mask = mask * (1.0 - np.clip(foreign, 0.0, 1.0))    # chừa bộ phận của người bên cạnh
    return mask, np.maximum(base_prot, own_prot), 'parsenet+parts'


def combine_masks(face_masks, prot_masks):
    """Quyết định pixel nào thuộc về ai ở vùng còn chồng nhau (chủ yếu là da má).

    Chỗ hai bên đòi ngang nhau thì trọng số của CẢ HAI về 0 -> giữ pixel gốc,
    thay vì để một bên xoá mất bộ phận của bên kia.
    """
    if len(face_masks) == 1:
        return face_masks
    F = np.stack(face_masks)
    score = F + PROTRUDE_BONUS * np.stack(prot_masks)
    out = []
    for i in range(len(face_masks)):
        other = np.max(np.delete(score, i, axis=0), axis=0)
        dom = score[i] - other                        # >0 = mình đòi mạnh hơn
        out.append(F[i] * np.clip(dom / CONFLICT_BAND, 0.0, 1.0))
    return out


def swap_all_faces(frame, assigned, target_faces, swapper, source_face):
    """Swap mọi người trong một frame rồi ghép lại, không ai đè lên ai.

    frame : khung GỐC (chưa swap gì). Gọi swapper tuần tự trên khung đã swap khiến
            lần thứ hai cắt crop từ vùng lần thứ nhất vừa ghi đè lên chính mặt nó.
    """
    if not assigned:
        return frame

    h, w = frame.shape[:2]
    fakes, faces_m, prots_m, centers, widths, yaws = [], [], [], [], [], []
    for label, idx in assigned.items():
        face = target_faces[idx]
        yaws.append(yaw_score(face))
        bgr_fake, M = swapper.get(frame, face, source_face[label], paste_back=False)
        IM = cv2.invertAffineTransform(M)
        mask, prot, _ = build_masks_for(frame, face, M, bgr_fake.shape[0])

        fakes.append(cv2.warpAffine(bgr_fake, IM, (w, h), borderValue=0.0).astype(np.float32))
        faces_m.append(cv2.warpAffine(mask, IM, (w, h), borderValue=0.0))
        prots_m.append(cv2.warpAffine(prot, IM, (w, h), borderValue=0.0))
        centers.append(face_center(face))
        widths.append(face_width(face))

    # Làm mềm viền TRƯỚC khi so sánh, để ranh giới chuyển tiếp mượt thay vì răng cưa
    feather = max(2.0, 0.04 * float(np.mean(widths)))
    faces_m = [cv2.GaussianBlur(m, (0, 0), sigmaX=feather) for m in faces_m]
    prots_m = [cv2.GaussianBlur(m, (0, 0), sigmaX=feather) for m in prots_m]

    # Đang chạm nhau -> chừa mũi + môi ra. Chỉ áp cho frame có tiếp xúc, để những frame
    # còn lại vẫn swap đầy đủ và giữ được danh tính.
    if NO_SWAP_MOUTH_NOSE and len(faces_m) > 1:
        mean_w = float(np.mean(widths))
        for i in range(len(faces_m)):
            touching = any(
                float(np.linalg.norm(centers[i] - centers[j])) / max(mean_w, 1e-6) < CONTACT_RATIO
                for j in range(len(faces_m)) if j != i
            )
            if touching:
                faces_m[i] = faces_m[i] * (1.0 - np.clip(prots_m[i], 0.0, 1.0))

    # Nghiêng sâu quá -> mờ dần rồi thôi không dán, để không có cảm giác dán mặt phẳng
    # lên cái đầu đang nghiêng.
    if YAW_FADE_ON:
        for i in range(len(faces_m)):
            wgt = yaw_weight(yaws[i])
            if wgt < 1.0:
                faces_m[i] = faces_m[i] * wgt
                prots_m[i] = prots_m[i] * wgt

    out = frame.astype(np.float32)
    for m, fk in zip(combine_masks(faces_m, prots_m), fakes):
        if float(m.max()) <= 0.0:
            continue
        out = m[..., None] * fk + (1.0 - m[..., None]) * out
    return np.clip(out, 0, 255).astype(np.uint8)


print(f'swap_all_faces sẵn sàng.  USE_PARSING={USE_PARSING}  '
      f'USE_PART_OWNERSHIP={USE_PART_OWNERSHIP}  '
      f'NO_SWAP_MOUTH_NOSE={NO_SWAP_MOUTH_NOSE} (ngưỡng chạm {CONTACT_RATIO}, '
      f'vùng mũi/môi từ template={MOUTH_NOSE_FROM_TEMPLATE})')
if YAW_FADE_ON:
    print(f'  Mặt nghiêng: dán đầy đủ tới yaw={YAW_FULL}, mờ dần, thôi hẳn từ yaw={YAW_NONE}')
else:
    print('  Mặt nghiêng: lớp mờ dần đang TẮT (xem phân bố yaw ở cell 6.2 rồi hãy bật)')

### 6.2 Chẩn đoán: đo số liệu thật từ video

Cell này **chỉ đọc, không tạo video**. Nó trả lời bốn câu hỏi, câu 3 và 4 là mới:

1. **Tracking có gán nhầm không?** `margin` = sim với ref của mình − sim với ref người kia. Về 0 hoặc âm nghĩa là chọn A/B đã thành tung đồng xu.
2. **Vùng paste-back mặc định tràn bao nhiêu?** Con số *nếu không sửa gì*, để đối chiếu.
3. **Ngưỡng "đang chạm" có kích hoạt không?** In khoảng cách 2 tâm mặt / bề ngang mặt của từng frame, so với `CONTACT_RATIO`, và đếm bao nhiêu frame vượt/không vượt. Nếu không frame nào dưới ngưỡng thì `NO_SWAP_MOUTH_NOSE` **chưa từng chạy** — và đó là lý do mũi/môi vẫn bị swap.
4. **Vùng mũi/môi có được nhận ra không?** Gọi thẳng `build_masks_for()` của cell 6.1b và in **diện tích `prot`** (vùng mũi + miệng + môi của chính khuôn mặt đó). `prot = 0` nghĩa là parsenet không nhận ra mũi/môi ở góc mặt này — lúc đó dù ngưỡng có kích hoạt cũng không có gì để trừ.

Hai câu cuối phân biệt được ba nguyên nhân khác nhau của cùng một triệu chứng "mũi môi vẫn bị swap": ngưỡng không kích hoạt / parsenet không thấy mũi môi / cả hai đều ổn nhưng vấn đề nằm chỗ khác.

Ảnh lưu ra tô **mask thật** (vùng sẽ được dán) theo màu nhãn, và khoanh **viền trắng quanh vùng mũi/môi** sẽ bị chừa ra khi chạm nhau. Không thấy viền trắng = parsenet không nhận ra vùng đó.

Chỉnh `DIAG_END` về vài trăm frame quanh cảnh hôn để chạy nhanh. Tracker luôn chạy từ frame 0 để trạng thái giống hệt lần xử lý thật.

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from insightface.utils import face_align

# ---------------- Cấu hình chẩn đoán ----------------
DIAG_START     = 0
DIAG_END       = None     # None = hết video
DIAG_SAVE_N    = 10       # lưu bao nhiêu frame đáng ngờ nhất ra ảnh
DIAG_DIR       = '/content/diag'
DIAG_DRAW_MASK = True     # tô mask thật (gọi build_masks_for của 6.1b)

os.makedirs(DIAG_DIR, exist_ok=True)
LABEL_COLORS = {'A': (255, 140, 0), 'B': (0, 140, 255)}      # BGR

assert 'build_masks_for' in globals() and 'SWAP_SIZE' in globals(), (
    'Chạy cell 6.1b (Ghép mặt) trước cell này.'
)


def paste_quad(face, size=SWAP_SIZE):
    """Ô vuông inswapper sẽ dán về NẾU dùng paste_back mặc định — để ĐO vấn đề
    của cách mặc định, KHÔNG phải mask pipeline đang dùng."""
    M = face_align.estimate_norm(face.kps, size)
    IM = cv2.invertAffineTransform(M)
    corners = np.array([[0, 0], [size, 0], [size, size], [0, size]], np.float32)
    quad = cv2.transform(corners.reshape(1, -1, 2), IM).reshape(-1, 2)
    w, h = float(np.ptp(quad[:, 0])), float(np.ptp(quad[:, 1]))
    ms = float(np.sqrt(max(w * h, 1.0)))
    k = max(int(ms) // 10, 10)
    ctr = quad.mean(0)
    return (ctr + (quad - ctr) * (1.0 - (k / 2.0) / (ms / 2.0))).astype(np.float32)


def bbox_poly(face):
    x1, y1, x2, y2 = [float(v) for v in face.bbox]
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], np.float32)


def covered_frac(poly, target):
    inter, _ = cv2.intersectConvexConvex(poly, target)
    area = cv2.contourArea(target)
    return float(inter / area) if area > 0 else 0.0


def fmt(v, nd=2):
    return '  --' if v is None else f'{v:+.{nd}f}'


def real_masks(frame, faces, assigned):
    """Mask THẬT pipeline dùng + vùng mũi/môi (prot), đưa về toạ độ khung hình."""
    h, w = frame.shape[:2]
    masks, prots, srcs, hairs = {}, {}, {}, {}
    for label, idx in assigned.items():
        f = faces[idx]
        M = face_align.estimate_norm(f.kps, SWAP_SIZE)
        IM = cv2.invertAffineTransform(M)
        m, p, src = build_masks_for(frame, f, M, SWAP_SIZE)
        masks[label] = cv2.warpAffine(m, IM, (w, h), borderValue=0.0)
        prots[label] = cv2.warpAffine(p, IM, (w, h), borderValue=0.0)
        srcs[label] = src
        # Diện tích parsenet gán là TÓC trong crop. Gần 0 ở mặt nghiêng nghĩa là nó
        # không thấy tóc -> tóc người nguồn có đường rò vào mask.
        try:
            aimg = cv2.warpAffine(frame, M, (SWAP_SIZE, SWAP_SIZE), borderValue=0.0)
            hairs[label] = float(np.isin(parse_crop(aimg), HAIR_LABELS).sum())
        except Exception:
            hairs[label] = None
    return masks, prots, srcs, hairs


def annotate(frame, faces, assigned, dbg, masks=None, prots=None):
    """mask thật = vùng tô màu | mũi/môi = viền trắng | bbox = khung đậm."""
    vis = frame.astype(np.float32)
    if masks:
        for label, m in masks.items():
            a = np.clip(m, 0.0, 1.0)[..., None] * 0.45
            vis = vis * (1 - a) + a * np.array(LABEL_COLORS[label], np.float32)
    vis = np.clip(vis, 0, 255).astype(np.uint8)

    if prots:
        for label, p in prots.items():
            binm = (np.clip(p, 0.0, 1.0) > 0.4).astype(np.uint8)
            if binm.any():
                cnts, _ = cv2.findContours(binm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(vis, cnts, -1, (255, 255, 255), 2)

    owner = {idx: label for label, idx in assigned.items()}
    for i, f in enumerate(faces):
        label = owner.get(i)
        col = LABEL_COLORS.get(label, (150, 150, 150))
        x1, y1, x2, y2 = f.bbox.astype(int)
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2)
        cv2.polylines(vis, [paste_quad(f).astype(np.int32)], True, col, 1)
        s = dbg['sim'][i]
        cv2.putText(vis,
                    f"{label or '?'} simA={fmt(s['A'])} simB={fmt(s['B'])} det={f.det_score:.2f}",
                    (x1, max(16, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1, cv2.LINE_AA)
    return vis


# ---------------- Quét video ----------------
cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'
ret, probe = cap.read()
assert ret, 'Không đọc được frame nào.'
fh, fw = probe.shape[:2]
cap.release()

cap = cv2.VideoCapture(source_video_path)
tracker = FaceTracker(fw, fh)
rows, keep = [], []
idx = -1
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
pbar = tqdm(total=(DIAG_END if DIAG_END is not None else total), unit='frame', desc='chẩn đoán')
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        idx += 1
        if DIAG_END is not None and idx > DIAG_END:
            break

        faces = app.get(frame)
        assigned, dbg = tracker.step(faces)     # luôn chạy từ frame 0
        pbar.update(1)
        if idx < DIAG_START:
            continue

        iA, iB = assigned.get('A'), assigned.get('B')
        row = {'frame': idx, 'n_faces': len(faces),
               'co_A': iA is not None, 'co_B': iB is not None}
        for label, i, other in (('A', iA, 'B'), ('B', iB, 'A')):
            row[f'yaw_{label}'] = yaw_score(faces[i]) if i is not None else None
            row[f'sim_{label}'] = dbg['sim'][i][label] if i is not None else None
            row[f'det_{label}'] = float(faces[i].det_score) if i is not None else None
            if (i is not None and dbg['sim'][i][label] is not None
                    and dbg['sim'][i][other] is not None):
                row[f'margin_{label}'] = dbg['sim'][i][label] - dbg['sim'][i][other]
            else:
                row[f'margin_{label}'] = None

        suspect = 0.0
        if iA is not None and iB is not None:
            fA, fB = faces[iA], faces[iB]
            ovBA = covered_frac(paste_quad(fB), bbox_poly(fA))
            ovAB = covered_frac(paste_quad(fA), bbox_poly(fB))
            d = float(np.linalg.norm(face_center(fA) - face_center(fB)))
            mw = (face_width(fA) + face_width(fB)) / 2.0
            ratio = d / mw if mw > 0 else None
            row['B_phu_len_mat_A'] = ovBA
            row['A_phu_len_mat_B'] = ovAB
            row['kc_2_mat_theo_be_ngang'] = ratio
            # Đúng điều kiện mà swap_all_faces dùng để quyết định chừa mũi/môi
            row['dang_cham'] = (ratio is not None and ratio < CONTACT_RATIO)
            suspect = max(ovBA, ovAB)
        else:
            row['B_phu_len_mat_A'] = row['A_phu_len_mat_B'] = None
            row['kc_2_mat_theo_be_ngang'] = None
            row['dang_cham'] = None
            if faces:
                suspect = 0.9

        for label in ('A', 'B'):
            m = row.get(f'margin_{label}')
            if m is not None and m < 0.25:
                suspect = max(suspect, 1.2 - m)

        rows.append(row)
        if suspect > 0.15:
            keep.append((suspect, idx, frame.copy(), faces, assigned, dbg))
            keep.sort(key=lambda t: -t[0])
            del keep[DIAG_SAVE_N:]
finally:
    pbar.close()
    cap.release()

print(f'\nĐã ghi số liệu {len(rows)} frame (video {fw}x{fh}).')

# ---------------- Ngưỡng chạm có kích hoạt không? ----------------
ratios = [r['kc_2_mat_theo_be_ngang'] for r in rows
          if r.get('kc_2_mat_theo_be_ngang') is not None]
print('\n' + '=' * 70)
print(f'NGƯỠNG CHẠM  (CONTACT_RATIO = {CONTACT_RATIO}, NO_SWAP_MOUTH_NOSE = {NO_SWAP_MOUTH_NOSE})')
print('=' * 70)
if not ratios:
    print('  Không frame nào detect đủ 2 người -> ngưỡng không bao giờ xét tới.')
else:
    fired = sum(1 for r in ratios if r < CONTACT_RATIO)
    a = np.array(ratios)
    print(f'  kc 2 tâm mặt / bề ngang mặt:  nhỏ nhất {a.min():.2f}  '
          f'trung vị {np.median(a):.2f}  lớn nhất {a.max():.2f}')
    print(f'  Số frame dưới ngưỡng (coi là đang chạm): {fired}/{len(ratios)} ({fired/len(ratios):.0%})')
    if fired == 0:
        print(f'  -> NGƯỠNG CHƯA TỪNG KÍCH HOẠT. Mũi/môi không bao giờ được chừa ra.')
        print(f'     Nâng CONTACT_RATIO lên khoảng {a.min() + 0.05:.2f} thì mới bắt được cảnh hôn.')
    else:
        print(f'  -> Ngưỡng CÓ kích hoạt ở {fired} frame.')

# ---------------- Mặt nghiêng bao nhiêu? ----------------
print()
print('=' * 70)
print(f'ĐỘ NGHIÊNG MẶT  (YAW_FADE_ON = {YAW_FADE_ON}, dán đủ tới {YAW_FULL}, '
      f'thôi hẳn từ {YAW_NONE})')
print('=' * 70)
print('  yaw = 0 là chính diện; càng lớn càng nghiêng. Bất biến với nghiêng đầu và cỡ mặt.')
for label in ('A', 'B'):
    ys = [r[f'yaw_{label}'] for r in rows if r.get(f'yaw_{label}') is not None]
    if not ys:
        print(f'  {label}: không có dữ liệu')
        continue
    a = np.array(ys)
    full = int((a <= YAW_FULL).sum())
    none = int((a >= YAW_NONE).sum())
    fade = len(a) - full - none
    print(f'  {label}: nhỏ nhất {a.min():.2f}  trung vị {np.median(a):.2f}  '
          f'lớn nhất {a.max():.2f}')
    print(f'      dán đầy đủ {full} frame | mờ dần {fade} frame | không dán {none} frame')
mx = [r.get('yaw_A') for r in rows if r.get('yaw_A') is not None]
if mx:
    hi = max(mx + [r['yaw_B'] for r in rows if r.get('yaw_B') is not None])
    if hi <= YAW_FULL:
        print(f'  -> Cả video đều dưới {YAW_FULL}: lớp mờ theo độ nghiêng CHƯA từng chạy.')
        print(f'     Muốn nó tác dụng thì hạ YAW_FULL xuống dưới {hi:.2f}.')

# ---------------- Mask thật + vùng mũi/môi ----------------
mask_rows = []
for _, i, frm, fcs, asg, dbgi in sorted(keep, key=lambda t: t[1]):
    masks = prots = None
    if DIAG_DRAW_MASK and asg:
        masks, prots, srcs, hairs = real_masks(frm, fcs, asg)
        rec = {'frame': i, 'nguon': '/'.join(sorted(set(srcs.values())))}
        for lb in ('A', 'B'):
            rec[f'mask_{lb}'] = float(masks[lb].sum()) if lb in masks else 0.0
            rec[f'prot_{lb}'] = float(prots[lb].sum()) if lb in prots else 0.0
            rec[f'hair_{lb}'] = hairs.get(lb)
        r0 = next((r for r in rows if r['frame'] == i), {})
        rec['ratio'] = r0.get('kc_2_mat_theo_be_ngang')
        rec['cham'] = r0.get('dang_cham')
        rec['yaw_A'], rec['yaw_B'] = r0.get('yaw_A'), r0.get('yaw_B')
        mask_rows.append(rec)
    cv2.imwrite(f'{DIAG_DIR}/frame_{i:05d}.jpg', annotate(frm, fcs, asg, dbgi, masks, prots))

print(f'\nĐã lưu {len(keep)} frame vào {DIAG_DIR}/')
print('  vùng tô màu = mask THẬT sẽ được dán | viền trắng = vùng mũi/môi (prot)')
print('  khung đậm = bbox | đường mảnh = ô paste-back mặc định của inswapper')

if mask_rows:
    print('\n' + '=' * 70)
    print('VÙNG MŨI/MÔI CÓ ĐƯỢC NHẬN RA KHÔNG?')
    print('=' * 70)
    print(f'  {"frame":>6} {"kc/bề ngang":>12} {"chạm":>6} {"prot A":>8} {"prot B":>8} '
          f'{"yaw A":>7} {"yaw B":>7} {"đậm A":>7} {"đậm B":>7}')
    for r in mask_rows:
        rt = f"{r['ratio']:.2f}" if r.get('ratio') is not None else '--'
        ch = ('CÓ' if r['cham'] else 'ko') if r.get('cham') is not None else '--'
        ya = f"{r['yaw_A']:.2f}" if r.get('yaw_A') is not None else '--'
        yb = f"{r['yaw_B']:.2f}" if r.get('yaw_B') is not None else '--'
        wa = f"{yaw_weight(r['yaw_A']):.2f}" if r.get('yaw_A') is not None else '--'
        wb = f"{yaw_weight(r['yaw_B']):.2f}" if r.get('yaw_B') is not None else '--'
        print(f'  {r["frame"]:>6} {rt:>12} {ch:>6} {r["prot_A"]:>8.0f} {r["prot_B"]:>8.0f} '
              f'{ya:>7} {yb:>7} {wa:>7} {wb:>7}')

    zero = sum(1 for r in mask_rows if r['prot_A'] == 0 or r['prot_B'] == 0)
    print()
    if zero == len(mask_rows):
        print('  -> prot = 0 ở MỌI frame: parsenet không nhận ra mũi/môi ở góc mặt này.')
        print('     Dù ngưỡng chạm có kích hoạt cũng KHÔNG có gì để trừ ra.')
        print(f'     Nới PART_MAX_DIST (hiện {PART_MAX_DIST}) hoặc kiểm tra parsenet '
              f'có thật sự chạy không.')
    elif zero:
        print(f'  -> {zero}/{len(mask_rows)} frame có một bên prot = 0: nhận ra không ổn định.')
    else:
        print('  -> prot > 0 ở mọi frame: mũi/môi được nhận ra bình thường.')
        print('     Nếu chúng vẫn bị swap thì nguyên nhân là ngưỡng chạm, xem mục trên.')

### 6.2b Đọc kết quả chẩn đoán

Tách riêng khỏi cell quét để phân tích lại mà không phải quét video lần nữa.

In [ ]:
import numpy as np

assert rows, 'Chưa có số liệu — chạy cell quét ở 6.2 trước.'

def col(name):
    return [r.get(name) for r in rows]

def vals(name):
    return [v for v in col(name) if v is not None]

n = len(rows)
print(f'Tổng số frame đã đo: {n}\n')

# ---- 1. Detector thấy mấy mặt mỗi frame ----
print('1) Số mặt detect được mỗi frame')
from collections import Counter
for k, v in sorted(Counter(col('n_faces')).items()):
    print(f'   {k} mặt : {v:>5} frame ({v/n:>5.1%})')

missing = [r for r in rows if r['n_faces'] > 0 and not (r['co_A'] and r['co_B'])]
print(f'\n   Frame có mặt nhưng THIẾU nhãn A hoặc B: {len(missing)} ({len(missing)/n:.1%})')

# ---- 2. Tracking có mơ hồ không? ----
print('\n2) Biên an toàn của tracking (margin = sim với ref của mình − sim với ref người kia)')
print('   margin cao = chắc chắn đúng người. margin ~0 hoặc âm = tung đồng xu.')
amb_low = amb_neg = 0
for label in ('A', 'B'):
    m = vals(f'margin_{label}')
    if not m:
        print(f'   {label}: không có dữ liệu')
        continue
    m = np.array(m)
    lo, neg = int((m < 0.10).sum()), int((m < 0).sum())
    amb_low += lo
    amb_neg += neg
    print(f'   {label}: min={m.min():+.3f}  p5={np.percentile(m,5):+.3f}  '
          f'trung vị={np.median(m):+.3f}  |  <0.10: {lo} frame  |  <0 (SAI HẲN): {neg} frame')

# ---- 3. Khi chỉ detect được 1 mặt thì nó thuộc về ai? ----
one = [r for r in rows if r['n_faces'] == 1]
print(f'\n3) Frame chỉ detect được 1 mặt: {len(one)}')
if one:
    c = Counter(r.get('mot_mat_gan_cho') for r in one)
    for k, v in c.items():
        print(f'   gán cho {k}: {v} frame')
    sa = [r['mot_mat_simA'] for r in one if r.get('mot_mat_simA') is not None]
    sb = [r['mot_mat_simB'] for r in one if r.get('mot_mat_simB') is not None]
    if sa and sb:
        print(f'   sim với ref A: trung vị {np.median(sa):+.3f}   '
              f'sim với ref B: trung vị {np.median(sb):+.3f}')

# ---- 4. Vùng paste-back có tràn sang mặt người kia không? ----
print('\n4) Chồng lấn vùng paste-back (chỉ tính frame có đủ cả 2 người)')
ovBA, ovAB = vals('B_phu_len_mat_A'), vals('A_phu_len_mat_B')
big = 0
if ovBA:
    a, b = np.array(ovBA), np.array(ovAB)
    big = int((np.maximum(a, b) > 0.30).sum())
    print(f'   ô paste của B phủ lên mặt A: trung vị {np.median(a):.1%}  tối đa {a.max():.1%}')
    print(f'   ô paste của A phủ lên mặt B: trung vị {np.median(b):.1%}  tối đa {b.max():.1%}')
    print(f'   frame có chồng lấn > 30%: {big} / {len(a)} ({big/len(a):.1%})')
    worst = sorted((r for r in rows if r.get('B_phu_len_mat_A') is not None),
                   key=lambda r: -max(r['B_phu_len_mat_A'], r['A_phu_len_mat_B']))[:8]
    print('\n   8 frame chồng lấn nặng nhất:')
    print(f'   {"frame":>7} {"B phủ A":>9} {"A phủ B":>9} {"kc/bề ngang":>12} '
          f'{"margin A":>9} {"margin B":>9}')
    for r in worst:
        ma = f"{r['margin_A']:+.2f}" if r.get('margin_A') is not None else '   --'
        mb = f"{r['margin_B']:+.2f}" if r.get('margin_B') is not None else '   --'
        print(f'   {r["frame"]:>7} {r["B_phu_len_mat_A"]:>8.1%} {r["A_phu_len_mat_B"]:>8.1%} '
              f'{r["kc_2_mat_theo_be_ngang"]:>12.2f} {ma:>9} {mb:>9}')
else:
    print('   (không có frame nào detect đủ 2 người)')

# ---- 5. Kết luận ----
print('\n' + '=' * 68)
print('KẾT LUẬN')
print('=' * 68)
if big and big / max(len(ovBA), 1) > 0.02:
    print(f'- TRÀN PASTE-BACK: {big} frame có ô dán phủ >30% mặt người kia.')
    print('  Vì vòng lặp luôn dán A trước rồi B sau, ở vùng chồng lấn B LUÔN đè lên A')
    print('  -> mặt người A hiện ra mặt của nguồn B. Khớp với triệu chứng một chiều.')
else:
    print('- Không thấy tràn paste-back đáng kể.')
if amb_neg:
    print(f'- TRACKING SAI HẲN: {amb_neg} frame có margin ÂM — mặt bị gán cho nhầm người.')
elif amb_low > max(3, 0.05 * n):
    print(f'- TRACKING MƠ HỒ: {amb_low} frame có margin < 0.10, ranh giới A/B mong manh.')
else:
    print(f'- Tracking có biên an toàn rõ ràng (chỉ {amb_low} frame margin < 0.10)'
          ' -> không phải nguyên nhân.')
if missing:
    print(f'- DETECTOR RỚT MẶT: {len(missing)} frame có mặt nhưng thiếu nhãn'
          ' -> người đó giữ nguyên mặt gốc ở frame đó.')
print('=' * 68)

# ---- 6. Biểu đồ theo thời gian ----
try:
    import matplotlib.pyplot as plt
    f = col('frame')
    fig, ax = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
    ax[0].step(f, col('n_faces'), where='post', lw=1)
    ax[0].set_ylabel('số mặt'); ax[0].set_yticks([0, 1, 2, 3]); ax[0].grid(alpha=.3)
    for label, c in (('A', 'tab:orange'), ('B', 'tab:blue')):
        ax[1].plot(f, col(f'margin_{label}'), lw=1, color=c, label=f'margin {label}')
    ax[1].axhline(0, color='r', lw=1, ls='--')
    ax[1].axhline(0.15, color='orange', lw=1, ls=':')
    ax[1].set_ylabel('margin'); ax[1].legend(loc='upper right', fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(f, [None if v is None else v * 100 for v in col('B_phu_len_mat_A')],
               lw=1, label='B phủ lên mặt A')
    ax[2].plot(f, [None if v is None else v * 100 for v in col('A_phu_len_mat_B')],
               lw=1, label='A phủ lên mặt B')
    ax[2].axhline(30, color='r', lw=1, ls='--')
    ax[2].set_ylabel('% chồng lấn'); ax[2].set_xlabel('frame')
    ax[2].legend(loc='upper right', fontsize=8); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'(bỏ qua biểu đồ: {type(e).__name__}: {e})')

### 6.3 Định nghĩa `process_frame()` — dùng chung cho preview và vòng lặp thật

In [ ]:
import os, sys, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

GFPGAN_PAD   = 0.4       # nới bbox bao nhiêu lần khi crop để làm nét
final_output = '/content/output_final.mp4'

# globals().get: cho phép bỏ qua HẲN mục 5 / 5b (không chạy cell nào ở đó) mà vẫn chạy được
# pipeline chính, thay vì NameError.
restorer = globals().get('restorer', None)
recolor  = globals().get('recolor', None)
print('Làm nét mặt:', 'BẬT' if restorer is not None else 'TẮT')
print('Nhuộm tóc  :', 'BẬT' if recolor is not None else 'TẮT',
      '' if recolor is None else f'({", ".join(sorted(recolor.target_lab))})')


def enhance_face_region(img, face, pad=GFPGAN_PAD):
    """Chỉ làm nét vùng quanh khuôn mặt vừa swap, KHÔNG chạy trên cả frame.

    Gọi restorer.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với app.get() vừa chạy), làm nét luôn cả những mặt
    trong nền không hề bị swap, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = restorer.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


STATS = dict(frames=0, swapped=0, detect=0.0, swap=0.0, enhance=0.0, hair=0.0)
STATS_LABEL = {l: dict(swapped=0, haired=0, stale=0) for l in LABELS}

# Vị trí + bề ngang mặt lần THẤY CUỐI của từng người. Phần chia tóc cần biết mặt mỗi người ở
# đâu; frame nào tracker không gán được ai thì dùng lại vị trí cũ, nhờ vậy tóc không bị dồn
# hết cho người còn lại chỉ vì một frame mất dấu.
_last_seen = {}


def process_frame(frame, verbose=False):
    """Xử lý MỘT frame. Dùng chung cho cả cell thử 1 frame lẫn vòng lặp cả video."""
    out = frame

    t0 = time.perf_counter()
    target_faces = app.get(frame)
    assigned = {}
    if target_faces:
        assigned, dbg = tracker.step(target_faces)
        if verbose:
            for label in dbg['late_init']:
                print(f'  khởi tạo danh tính {label}')
    t1 = time.perf_counter()
    STATS['detect'] += t1 - t0

    if assigned:
        # Ghép TẤT CẢ mặt cùng lúc TỪ FRAME GỐC (xem mục 6.1b). Gọi swapper tuần tự trên
        # frame đã swap làm người dán sau đè lên người dán trước, đồng thời khiến swap thứ
        # hai cắt crop từ vùng đã bị swap thứ nhất ghi đè.
        out = swap_all_faces(frame, assigned, target_faces, swapper, source_face)
        t2 = time.perf_counter()
        STATS['swap'] += t2 - t1

        if restorer is not None:
            for idx in assigned.values():
                out = enhance_face_region(out, target_faces[idx])
        STATS['enhance'] += time.perf_counter() - t2
        STATS['swapped'] += 1
        for label in assigned:
            STATS_LABEL[label]['swapped'] += 1

    # Cập nhật vị trí lần thấy cuối, rồi dựng danh sách người cho phần chia tóc.
    for label, idx in assigned.items():
        f = target_faces[idx]
        _last_seen[label] = {'center': face_center(f), 'width': face_width(f), 'kps': f.kps}

    t3 = time.perf_counter()
    if recolor is not None:
        people = {}
        for label in LABELS:
            if label in assigned:
                f = target_faces[assigned[label]]
                people[label] = {'center': face_center(f), 'width': face_width(f),
                                 'kps': f.kps}
            elif label in _last_seen:
                # Vị trí cũ: đủ để CHIA mask (người ta không dịch chuyển tức thời), nhưng
                # bỏ kps đi - landmark cũ đặt sai chỗ thì tầng tóc mai sẽ chừa nhầm vùng
                # mắt/mũi/miệng, tệ hơn là không chạy nó.
                people[label] = {'center': _last_seen[label]['center'],
                                 'width': _last_seen[label]['width'], 'kps': None}
                STATS_LABEL[label]['stale'] += 1
        out, done = recolor.apply(out, people)
        for label in done:
            STATS_LABEL[label]['haired'] += 1
    STATS['hair'] += time.perf_counter() - t3

    STATS['frames'] += 1
    return out


def reset_state():
    """Đưa STATS, tracker và trạng thái thời gian về 0 (gọi trước mỗi lần chạy lại).

    Tracker PHẢI được dựng lại: nó giữ embedding tham chiếu và vị trí cũ, chạy lại video mà
    không reset thì frame đầu của lần chạy sau vẫn mang danh tính của lần chạy trước.
    """
    global tracker
    for k in STATS:
        STATS[k] = 0 if isinstance(STATS[k], int) else 0.0
    for l in STATS_LABEL:
        for k in STATS_LABEL[l]:
            STATS_LABEL[l][k] = 0
    _last_seen.clear()
    tracker = FaceTracker(VIDEO_W, VIDEO_H)
    if recolor is not None:
        recolor.reset()
        for l in list(recolor.n_applied):
            recolor.n_applied[l] = 0


# Kích thước video: cần cho FaceTracker (bán kính chuẩn hoá khoảng cách) nên đọc ngay ở đây,
# trước cả vòng lặp, để cell thử-1-frame cũng dùng được.
_cap = cv2.VideoCapture(source_video_path)
assert _cap.isOpened(), f'Không mở được video: {source_video_path}'
_ok, _f0 = _cap.read()
_cap.release()
assert _ok, 'Không đọc được frame nào từ video.'
VIDEO_H, VIDEO_W = _f0.shape[:2]
tracker = FaceTracker(VIDEO_W, VIDEO_H)

print(f'Video: {VIDEO_W}x{VIDEO_H}')
print('Đã định nghĩa process_frame(). Chạy cell 6.4 để thử 1 frame trước khi làm cả video.')

### 6.4 Thử một frame

Xem **mặt swap đúng người chưa** và **tóc chia đúng người chưa** (ảnh thứ hai tô màu vùng tóc: A cam, B xanh). Vài giây một vòng, thay vì chờ cả video.

In [ ]:
# Thử ĐÚNG MỘT frame: xem mặt swap đúng người chưa, tóc chia đúng người chưa.
# Chỉnh ở mục 0 -> chạy lại mục 0 -> chạy lại mục 5b (nạp màu mới) -> chạy lại cell này.
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

PREVIEW_AT = 0.5      # lấy frame ở đâu trong video: 0.0 = đầu, 0.5 = giữa, 0.95 = gần cuối

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if n_total > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n_total * PREVIEW_AT))
ok, frame = cap.read()
if not ok:
    # Seek thất bại với một số container -> quay lại đọc tuần tự từ đầu.
    cap.release()
    cap = cv2.VideoCapture(source_video_path)
    ok, frame = cap.read()
cap.release()
assert ok, 'Không đọc được frame nào từ video.'

before = frame.copy()

reset_state()
# Chạy 2 lần trên cùng 1 frame: lần đầu khởi tạo danh tính + EMA mask, lần hai mới đúng trạng
# thái mà video thật sẽ có. Không làm vậy thì preview khác kết quả cuối một chút.
process_frame(frame.copy())
after = process_frame(frame.copy(), verbose=True)

strip = np.hstack([before, after])
sc = min(1.0, 1200 / strip.shape[1])
if sc < 1.0:
    strip = cv2.resize(strip, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
print('TRƯỚC  |  SAU')
cv2_imshow(strip)

# ---- Vẽ VÙNG TÓC CỦA TỪNG NGƯỜI. Nhìn thẳng vào đây là biết mask chia đúng hay sai:
# ---- A tô cam, B tô xanh. Cam trùm sang đầu B (hay ngược lại) = chia sai.
LABEL_TINT = {'A': (0, 140, 255), 'B': (255, 140, 0)}      # BGR: A cam, B xanh
if recolor is not None and recolor.last_alpha:
    ov = before.astype(np.float32)
    for label, a in recolor.last_alpha.items():
        bx1, by1, bx2, by2 = recolor.last_box[label]
        tint = np.zeros((by2 - by1, bx2 - bx1, 3), np.float32)
        tint[:] = LABEL_TINT.get(label, (255, 255, 255))
        a3 = np.clip(a, 0, 1)[..., None] * 0.6
        ov[by1:by2, bx1:bx2] = ov[by1:by2, bx1:bx2] * (1 - a3) + tint * a3
    ov = np.clip(ov, 0, 255).astype(np.uint8)
    if sc < 1.0:
        ov = cv2.resize(ov, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
    print()
    print('VÙNG TÓC ĐANG NHUỘM:  A = CAM,  B = XANH')
    cv2_imshow(ov)

print()
for label in LABELS:
    s = STATS_LABEL[label]
    col = recolor.target_bgr.get(label) if recolor is not None else None
    print(f'  {label}: swap {"CÓ" if s["swapped"] else "KHÔNG"}'
          f' | nhuộm {"CÓ" if s["haired"] else "KHÔNG"}'
          f' | màu {bgr_to_hex(col) if col is not None else "(không đổi)"}'
          f'{"  | dùng vị trí cũ để chia tóc" if s["stale"] else ""}')

print()
print('Thấy gì thì chỉnh nấy (ở mục 0, rồi chạy lại mục 0 + 5b + cell này):')
print('  MẶT bị đảo A/B                     -> đổi thứ tự 2 ảnh upload ở mục 3')
print('  màu của A lấn sang tóc B (hoặc ngược) -> giảm HAIR_SPLIT_BAND (0.15)')
print('  thấy đường ranh giữa mảng tóc      -> tăng HAIR_SPLIT_BAND (0.5)')
print('  phần cuối tóc dài không ăn màu     -> tăng HAIR_REACH (6.0)')
print('  màu không hiện ra trên tóc tối     -> tăng HAIR_LIGHTNESS (1.0)')
print('  tóc bệt, phẳng, mất sợi            -> tăng HAIR_DETAIL (1.5)')
print('  còn mảng màu cũ ở đỉnh đầu         -> giảm HAIR_KEEP_HIGHLIGHT (0.2)')
print('  màu lem sang nền / da / vai        -> nâng HAIR_CONF, vd (0.5, 0.8)')
print('  sợi tóc mai trước mặt vẫn màu cũ   -> tăng HAIR_WISPS (0.9)')
print('  còn vệt sáng ở đường viền tóc      -> HAIR_EDGE_SMART = True')

### 6.5 Thử nhiều cặp màu trên một frame

In [ ]:
# Thử NHIỀU CẶP màu (A, B) cùng lúc trên đúng một frame, để chọn bằng mắt thay vì đoán.
# Phần swap mặt chỉ chạy MỘT lần rồi dùng lại cho mọi cặp - nó không phụ thuộc màu tóc.
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# Mỗi phần tử là một cặp (màu của A, màu của B). Nhận tên trong bảng màu, mã hex, hoặc None.
TRY_PAIRS = [
    ('nâu hạt dẻ',   'nâu tây'),
    ('vàng đồng',    'nâu socola'),
    ('bạch kim',     'nâu đen'),
    ('đỏ rượu',      'vàng mật ong'),
    ('tím khói',     'bạch kim'),
]
TRY_AT = 0.5      # vị trí frame trong video, giống PREVIEW_AT

if recolor is None:
    print('USE_HAIR = False (hoặc mediapipe không dùng được) -> không có gì để thử.')
else:
    cap = cv2.VideoCapture(source_video_path)
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n_total * TRY_AT))
    ok, base = cap.read()
    if not ok:
        cap.release(); cap = cv2.VideoCapture(source_video_path); ok, base = cap.read()
    cap.release()
    assert ok, 'Không đọc được frame nào từ video.'

    # Swap mặt 1 lần, đồng thời lấy vị trí 2 người để chia tóc.
    reset_state()
    faces = app.get(base)
    assigned, _ = tracker.step(faces) if faces else ({}, None)
    swapped = swap_all_faces(base, assigned, faces, swapper, source_face) if assigned else base
    if assigned and restorer is not None:
        for idx in assigned.values():
            swapped = enhance_face_region(swapped, faces[idx])

    people = {}
    for label, idx in assigned.items():
        f = faces[idx]
        people[label] = {'center': face_center(f), 'width': face_width(f), 'kps': f.kps}
    if len(people) < 2:
        print(f'CẢNH BÁO: frame này chỉ gán được {sorted(people) or "không ai"} -> '
              f'phép chia tóc không đại diện. Thử TRY_AT khác.')

    saved = dict(recolor.target_bgr)          # giữ lại màu đã chọn ở mục 0
    tiles, labels_txt = [swapped.copy()], ['(goc)']
    for ca, cb in TRY_PAIRS:
        for label, want in (('A', ca), ('B', cb)):
            if want is None:
                recolor.set_target(label, None)
            else:
                recolor.set_target(label, hex_to_bgr(HAIR_PALETTE.get(want, want))[0, 0])
        recolor.reset()
        img = swapped.copy(); recolor.apply(img, people)      # lần 1: nạp EMA mask
        img = swapped.copy(); recolor.apply(img, people)      # lần 2: đúng trạng thái thật
        tiles.append(img)
        labels_txt.append(f'A={ca} / B={cb}')

    # trả lại đúng màu đã chọn ở mục 0
    for label in LABELS:
        recolor.set_target(label, saved.get(label))
    recolor.reset()

    # Cắt quanh CẢ HAI đầu cho dễ so sánh.
    if people:
        cs = np.array([p['center'] for p in people.values()])
        wd = float(np.mean([p['width'] for p in people.values()]))
        x1 = max(0, int(cs[:, 0].min() - 2.0 * wd)); x2 = min(base.shape[1], int(cs[:, 0].max() + 2.0 * wd))
        y1 = max(0, int(cs[:, 1].min() - 2.0 * wd)); y2 = min(base.shape[0], int(cs[:, 1].max() + 2.2 * wd))
        if x2 - x1 > 40 and y2 - y1 > 40:
            tiles = [t[y1:y2, x1:x2] for t in tiles]

    tw = 300
    tiles = [cv2.resize(t, (tw, max(1, int(t.shape[0] * tw / t.shape[1]))),
                        interpolation=cv2.INTER_AREA) for t in tiles]
    th = min(t.shape[0] for t in tiles)
    tiles = [t[:th] for t in tiles]

    cols = 3
    grid = []
    for i in range(0, len(tiles), cols):
        row = tiles[i:i + cols]
        row += [np.zeros((th, tw, 3), np.uint8)] * (cols - len(row))
        grid.append(np.hstack(row))
    cv2_imshow(np.vstack(grid))
    print('Thứ tự trái->phải, trên->xuống:')
    for i, t in enumerate(labels_txt):
        print(f'  {i}. {t}')
    print()
    print('Chọn được rồi thì đặt HAIR_COLOR_A / HAIR_COLOR_B ở mục 0, chạy lại mục 0 + 5b, '
          'rồi chạy mục 6.5.')

### 6.6 Chạy toàn bộ video

In [ ]:
import os, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps <= 0:       # 0.0 hoặc NaN với một số container
    print('Không đọc được fps từ video, mặc định 25.')
    fps = 25.0

# CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho progress bar.
# Vòng lặp đọc tới khi hết frame thật sự, thay vì range(total_frames) (đếm thiếu = cụt đuôi video).
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

# Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có metadata
# rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg nhận rawvideo sai size.
ret, frame = cap.read()
assert ret, 'Không đọc được frame nào từ video.'
height, width = frame.shape[:2]

print(f'Video: {width}x{height} @ {fps:.2f}fps, ~{total_frames} frames')

# Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
ffmpeg_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}', '-r', f'{fps}', '-i', 'pipe:0',
    '-i', source_video_path,
    '-map', '0:v:0', '-map', '1:a:0?',        # '?' = không có audio thì bỏ qua, không lỗi
    '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
    '-pix_fmt', 'yuv420p',                    # để trình duyệt/IPython.display.Video phát được
    '-c:a', 'aac', '-shortest',
    final_output,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

reset_state()                                  # xoá số liệu + dựng lại tracker sau cell preview
pbar = tqdm(total=total_frames, unit='frame')
t_start = time.perf_counter()
rc = None
try:
    while frame is not None:
        result_frame = process_frame(frame, verbose=True)

        proc.stdin.write(np.ascontiguousarray(result_frame).tobytes())
        pbar.update(1)

        ret, frame = cap.read()
        if not ret:
            frame = None
finally:
    # Không release trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và file mp4 hỏng.
    pbar.close()
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    rc = proc.wait()

wall = time.perf_counter() - t_start
assert rc == 0, f'ffmpeg thất bại (exit code {rc}) - xem log lỗi ở trên.'

n  = max(STATS['frames'], 1)
ns = max(STATS['swapped'], 1)

print()
print(f'Xong: {STATS["frames"]} frame. Có ít nhất 1 người được swap ở {STATS["swapped"]} '
      f'frame ({STATS["swapped"] / n:.0%}).')
print()
print(f'{"":4} {"swap mặt":>18} {"nhuộm tóc":>18} {"chia tóc theo vị trí cũ":>26}')
for label in LABELS:
    s = STATS_LABEL[label]
    print(f'{label:4} {s["swapped"]:8d} ({s["swapped"] / n:3.0%})'
          f' {s["haired"]:9d} ({s["haired"] / n:3.0%})'
          f' {s["stale"]:17d} ({s["stale"] / n:3.0%})')
print('  swap thấp  -> tracker không gán được người đó; xem mục 6.2/6.3 để biết vì sao.')
print('  nhuộm thấp -> không thấy tóc người đó (quay xa, bị che), hoặc HAIR_CONF quá cao.')
print('  "vị trí cũ" cao -> mặt hay mất dấu; phép chia tóc vẫn chạy nhưng kém chính xác dần.')
print(f'Output: {final_output}')

# ---- Thời gian từng bước: để biết cái nào tốn, thay vì đoán ----
print()
print('Thời gian trung bình mỗi frame:')
print(f'  detect + tracking : {STATS["detect"] / n * 1000:7.1f} ms')
print(f'  swap (2 người)    : {STATS["swap"] / ns * 1000:7.1f} ms  (tính trên frame có swap)')
print(f'  làm nét           : {STATS["enhance"] / ns * 1000:7.1f} ms'
      f'{"" if restorer is not None else "      (TẮT)"}')
print(f'  nhuộm tóc         : {STATS["hair"] / n * 1000:7.1f} ms  (tính trên MỌI frame)'
      f'{"" if recolor is not None else "   (TẮT)"}')

busy = STATS['detect'] + STATS['swap'] + STATS['enhance'] + STATS['hair']
if restorer is not None:
    print(f'-> làm nét chiếm {STATS["enhance"] / max(busy, 1e-9):.0%} thời gian xử lý '
          f'(USE_GFPGAN = False để bỏ).')
if recolor is not None:
    print(f'-> nhuộm tóc chiếm {STATS["hair"] / max(busy, 1e-9):.0%} thời gian xử lý.')

print()
print(f'Tổng: {wall:.1f}s cho {STATS["frames"]} frame ({wall / n * 1000:.0f} ms/frame, '
      f'{n / max(wall, 1e-9):.1f} fps xử lý).')

## 7. Kiểm tra kết quả

Audio đã được ghép ngay trong cell trên (ffmpeg nhận frame qua pipe và mux luôn audio gốc trong cùng một pass), nên ở đây chỉ cần xác nhận file xuất ra hợp lệ.

In [ ]:
import os

assert os.path.exists(final_output) and os.path.getsize(final_output) > 0, \
    'Không tạo được video output — chạy lại cell xử lý video ở mục 6.'
print(f'{final_output}  —  {os.path.getsize(final_output) / 1e6:.1f} MB\n')

# Xác nhận có stream video (và audio, nếu video gốc có audio)
!ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,r_frame_rate,duration \
  -of default=noprint_wrappers=1 {final_output}

## 8. Xem kết quả

In [ ]:
import os
from IPython.display import Video, display

size_mb = os.path.getsize(final_output) / 1e6
if size_mb > 50:
    # embed=True nhét toàn bộ file dưới dạng base64 vào output của notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài. Trường hợp đó thì tải về xem thay vì preview inline.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về máy.')
else:
    display(Video(final_output, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Giới hạn / Cách chỉnh

### Vòng chỉnh cho nhanh

Đừng chạy cả video để thử. Vòng đúng: **sửa mục 0 → chạy lại cell mục 0 → chạy lại cell mục 5b**
(để nạp màu mới) **→ chạy lại cell 6.4**. Vài giây một vòng. Muốn so nhiều cặp màu thì dùng
cell **6.5** — nó nhuộm 5 cặp lên cùng một frame và xếp thành lưới.

Thử thêm vài mốc `PREVIEW_AT` (0.1 / 0.5 / 0.9) để bắt cả những đoạn hai người áp sát nhau —
đó là chỗ khó nhất cho cả swap lẫn chia tóc.

### Bảng tra: thấy gì thì chỉnh gì

| Hiện tượng | Chỉnh |
|---|---|
| **Mặt** bị đảo A/B | đổi thứ tự 2 ảnh upload ở mục 3 (không cần sửa code) |
| Màu tóc của A lấn sang đầu B | giảm `HAIR_SPLIT_BAND` (0.15) |
| Có đường ranh màu chạy qua mảng tóc | tăng `HAIR_SPLIT_BAND` (0.5) |
| Phần cuối tóc dài không ăn màu | tăng `HAIR_REACH` (6.0) |
| Tóc người thứ ba trong khung cũng bị nhuộm | giảm `HAIR_REACH` (3.0) |
| Màu không hiện ra trên tóc tối | tăng `HAIR_LIGHTNESS` (1.0) |
| Tóc bệt, phẳng, "như quét sơn" | tăng `HAIR_DETAIL` (1.5) |
| Còn mảng màu cũ ở đỉnh đầu / chỗ rẽ ngôi | giảm `HAIR_KEEP_HIGHLIGHT` (0.2) |
| Màu lem sang nền / áo / vai | nâng `HAIR_CONF` (0.5, 0.8) |
| Sợi tóc mai trước mặt vẫn màu cũ | tăng `HAIR_WISPS` (0.9) |
| Còn vệt sáng ở đường viền tóc | `HAIR_EDGE_SMART = True` |
| Chỉ muốn nhuộm một người | đặt màu người kia là `None` |

### Giới hạn riêng của việc chia tóc A/B

- **Phép chia là suy đoán hình học, không phải nhận dạng.** Nó gán theo "gần mặt ai hơn". Tóc A
  buông qua vai B thì bị tính là tóc B — không có thông tin nào trong mask để biết sợi tóc đó
  mọc từ đầu ai. Đây là giới hạn cốt lõi, không chỉnh tham số nào chữa được.
- **Hai đầu áp sát nhau** (đúng cảnh hôn) là lúc phép chia khó nhất: hai mặt cách nhau chưa tới
  một bề ngang mặt nên ranh giới chạy ngay giữa vùng tóc chồng nhau. Xem ảnh tô màu ở cell 6.4
  để biết nó chia thế nào trên đúng video của bạn.
- **Frame mất dấu một người** dùng vị trí lần thấy cuối. Người đó đứng yên thì không sao; nếu
  họ di chuyển nhiều trong lúc mất dấu thì ranh giới lệch dần. Cột `chia tóc theo vị trí cũ`
  ở mục 6.6 cho biết chuyện này xảy ra bao nhiêu frame.
- **Cả hai đều mất dấu** → không nhuộm frame đó (an toàn hơn là đoán).

### Giới hạn của phần nhuộm (giống bản 1 người)

- **Tóc gần đen nhuộm sang màu sáng sẽ phẳng hơn tóc vốn đã sáng.** Pixel tối gần như không còn
  biến thiên độ sáng để giữ — ngoài đời cũng phải tẩy tóc trước mới nhuộm sáng được.
  `HAIR_DETAIL` bù lại được một phần, nhưng không tạo ra chi tiết vốn không có trong video.
- **Mép tóc** là chỗ khó nhất: pixel ở đúng biên là màu PHA giữa tóc và nền, mà mask 256×256
  quá thô để biết tỉ lệ pha. `HAIR_EDGE_SMART` đọc tỉ lệ đó từ độ sáng của ảnh, nhưng **bất
  lực khi tóc và nền sáng xấp xỉ nhau** (tóc đen trên nền tối) — lúc đó nó tự quay về dùng mask.
- **MediaPipe nhầm lớp** khi tóc trùng màu nền, khi đội mũ, hoặc tóc bị tay che. Xem trực tiếp
  mask ở cell mục 5b trước khi chạy cả video.

### Giới hạn của phần swap 2 người (giống bản kiss)

- **Lúc hôn, mũi và môi được CHỪA RA** (`NO_SWAP_MOUTH_NOSE` ở mục 6.1b) vì đó là vùng biến
  dạng nặng nhất. Đánh đổi: mũi/môi mang danh tính rất nặng, bỏ đi thì bớt giống người nguồn.
  Đặt `False` để so hai bản trên cùng video.
- **Mặt nghiêng sâu**: `inswapper_128` học chủ yếu trên mặt gần chính diện nên nghiêng mạnh thì
  nó làm phẳng khuôn mặt. Lớp mờ dần theo độ nghiêng (`YAW_FADE_ON`) mặc định **TẮT** — số đo
  thật cho thấy với ngưỡng ban đầu nó tắt swap trên hơn nửa số frame. Xem phân bố yaw ở mục
  6.2 rồi hãy bật và chọn ngưỡng theo số đo.
- **Tracker gán nhầm A/B** thì mục 6.2/6.3 sẽ chỉ ra: nó in `margin` = (độ giống với chính
  mình) − (độ giống với người kia). Margin âm ở frame nào là gán nhầm ở đó.

### Notebook nào cho việc nào

| Muốn | Dùng |
|---|---|
| 1 người, chỉ thay mặt | `video_face_swap_1nguoi.ipynb` |
| 1 người, thay mặt + đổi màu tóc | `video_face_swap_1nguoi_mau_toc.ipynb` |
| 2 người (cảnh hôn), chỉ thay mặt | `video_face_swap_kiss.ipynb` |
| **2 người, thay mặt + đổi màu tóc từng người** | **notebook này** |